In [0]:
%pip install geopy
dbutils.library.restartPython()

In [0]:



# from geopy.geocoders import Nominatim
# from geopy.extra.rate_limiter import RateLimiter

# # Initialize geolocator
# geolocator = Nominatim(user_agent="vf_geocoder", timeout=10)

# # Rate limiter (VERY IMPORTANT)
# geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

# # In-memory cache to avoid repeated API calls
# _geo_cache = {}

In [0]:
# Databricks notebook source
# MAGIC %pip install geopy
# MAGIC dbutils.library.restartPython()

# COMMAND ----------
# MAGIC %md
# MAGIC # 02 — Silver Layer (v5)
# MAGIC
# MAGIC **Fixes applied vs v4 (all issues from review docs):**
# MAGIC
# MAGIC | Fix ID | Description |
# MAGIC |--------|-------------|
# MAGIC | SIL-V5-1  | **24/7 false-positive bug** — hard guard rejects numeric matches near time/social tokens |
# MAGIC | SIL-V5-2  | **Canonical field aliases** — projection layer adds exact PDF-spec names (officialWebsite, address_stateOrRegion, etc.) |
# MAGIC | SIL-V5-3  | **Stronger capability junk filter** — catches "NHIS accredited", "has telephone number", "has email address", NHIS patterns, admin statements |
# MAGIC | SIL-V5-4  | **Procedure/equipment mining** — secondary pass mines procedures/equipment from description when arrays are empty |
# MAGIC | SIL-V5-5  | **Row-level citation columns** — procedure_citations, equipment_citations, capability_citations as structured provenance |
# MAGIC | SIL-V5-6  | **doc_text + is_rag_ready** — added (required by RAG pipeline; was missing from active v4) |
# MAGIC | SIL-V5-7  | **Content flag columns** — has_procedures, has_equipment, has_capabilities, has_specialties, procedure_count, etc. |
# MAGIC | SIL-V5-8  | **Confidence scores** — facility_type_confidence, region_confidence, geo_confidence, geo_source |
# MAGIC | SIL-V5-9  | **Row quality flags** — row_quality_flags array (e.g. "low_doctors_coverage", "capability_contamination_risk") |
# MAGIC | SIL-V5-10 | **Deterministic extraction order** — structured field > mined > inferred fallback (no silent overrides) |

# COMMAND ----------
# MAGIC %md ## 0. Imports & Config

# COMMAND ----------

import json, re, time
from typing import Optional, List
from datetime import datetime, UTC
from functools import reduce
import operator as op_module
from difflib import get_close_matches
from concurrent.futures import ThreadPoolExecutor

import pandas as pd
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (
    StringType, IntegerType, FloatType, BooleanType,
    ArrayType, DoubleType, StructType, StructField,
)
from pyspark.sql.window import Window
from pyspark.sql.functions import pandas_udf

spark = SparkSession.builder.getOrCreate()
print(f"Run: {datetime.now(UTC).isoformat()}")

# COMMAND ----------

class Config:
    BRONZE    = "virtue_foundation.ghana.bronze_facilities_raw"
    SILVER    = "virtue_foundation.ghana.silver_facilities_cleaned"
    GEO_CACHE = "virtue_foundation.ghana.geo_cache"
    DEFAULT_LAT = 7.9465    # Ghana centroid
    DEFAULT_LON = -1.0232

cfg = Config()

spark.sql("CREATE DATABASE IF NOT EXISTS virtue_foundation.ghana")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {cfg.GEO_CACHE}
    (geo_key STRING, latitude DOUBLE, longitude DOUBLE)
    USING DELTA
""")

# COMMAND ----------
# MAGIC %md ## 1. Read Bronze

# COMMAND ----------

bronze = spark.table(cfg.BRONZE)
print(f"Bronze rows: {bronze.count():,}  columns: {len(bronze.columns)}")

# COMMAND ----------
# MAGIC %md ## 2. JSON Array Parser UDF

# COMMAND ----------

def normalize_text(x):
    if not x:
        return ""
    x = str(x).lower()
    x = re.sub(r"[^\w\s]", " ", x)
    x = re.sub(r"\s+", " ", x)
    return x.strip()

def _safe_json_parse(text: Optional[str]) -> List[str]:
    """Parse JSON arrays that may have double-escaped quotes from CSV export."""
    if text is None or str(text).strip() in ("", "null", "[]", "None"):
        return []
    text = str(text).strip()
    for attempt in [text, text.replace('""', '"')]:
        if attempt.startswith('"[') and attempt.endswith(']"'):
            attempt = attempt[1:-1]
        try:
            result = json.loads(attempt)
            if isinstance(result, list):
                return [str(x).strip() for x in result if x is not None and str(x).strip()]
            return [str(result)]
        except json.JSONDecodeError:
            pass
    cleaned = text.strip('"').strip("'")
    if "," in cleaned:
        return [x.strip().strip('"').strip("'") for x in cleaned.split(",") if x.strip()]
    return [cleaned] if cleaned else []

@pandas_udf(ArrayType(StringType()))
def safe_json_udf(col: pd.Series) -> pd.Series:
    return col.apply(_safe_json_parse)



# COMMAND ----------
# MAGIC %md ## 3. Region Normalisation Constants

# COMMAND ----------

REGION_NORM_MAP = {
    "Ashanti Region": "Ashanti",        "Ashanti": "Ashanti",
    "Greater Accra Region": "Greater Accra", "Greater Accra": "Greater Accra",
    "Western Region": "Western",        "Western": "Western",
    "Eastern Region": "Eastern",        "Eastern": "Eastern",
    "Volta Region": "Volta",            "Volta": "Volta",
    "Central Region": "Central",        "Central": "Central",
    "Brong Ahafo Region": "Brong-Ahafo","Brong-Ahafo Region": "Brong-Ahafo",
    "Brong Ahafo": "Brong-Ahafo",       "Brong-Ahafo": "Brong-Ahafo",
    "Bono Region": "Brong-Ahafo",
    "Northern Region": "Northern",      "Northern": "Northern",
    "Upper East Region": "Upper East",  "Upper East": "Upper East",
    "Upper West Region": "Upper West",  "Upper West": "Upper West",
    "Oti Region": "Oti",                "Oti": "Oti",
    "Bono East Region": "Bono East",    "Bono East": "Bono East",
    "Ahafo Region": "Ahafo",            "Ahafo": "Ahafo",
    "Savannah Region": "Savannah",      "Savannah": "Savannah",
    "North East Region": "North East",  "North East": "North East",
    "Western North Region": "Western North", "Western North": "Western North",
    "Asutifi South": "Ahafo",           "Asante": "Ashanti",
    "Central Tongu District": "Volta",  "Accra": "Greater Accra",
    "Kumasi": "Ashanti",                "Takoradi": "Western",
    "Tamale": "Northern",               "North Legon": "Greater Accra",
    "Cape Coast": "Central",            "Cape Coast Municipal": "Central",
    "Koforidua": "Eastern",             "Ho": "Volta",
    "Hohoe": "Volta",                   "Techiman": "Bono East",
    "Techiman Municipal": "Bono East",  "Sunyani": "Brong-Ahafo",
    "Bono": "Brong-Ahafo",              "Sekondi": "Western",
    "Sekondi-Takoradi": "Western",      "Bolgatanga": "Upper East",
    "Wa": "Upper West",                 "Goaso": "Ahafo",
    "Damongo": "Savannah",              "Accra Region": "Greater Accra",
    "Asante Region": "Ashanti",         "Elmina": "Central",
    # FIX SIL-V5: agona nkwanta is CENTRAL, not Oti
    "Agona Nkwanta": "Central",         "agona nkwanta": "Central",
}

VALID_REGIONS = {
    "Ashanti", "Greater Accra", "Western", "Eastern", "Central",
    "Volta", "Northern", "Upper East", "Upper West",
    "Oti", "Bono East", "Ahafo", "Savannah", "North East",
    "Western North", "Brong-Ahafo",
}

def _norm_region_field(r: Optional[str]) -> Optional[str]:
    if not r or str(r).strip().lower() in ("", "null", "none"):
        return None
    r = str(r).strip()
    if r in REGION_NORM_MAP:              return REGION_NORM_MAP[r]
    if r.title() in REGION_NORM_MAP:      return REGION_NORM_MAP[r.title()]
    stripped = re.sub(r"\s*[Rr]egion$", "", r).strip()
    if stripped in REGION_NORM_MAP:       return REGION_NORM_MAP[stripped]
    if stripped.title() in REGION_NORM_MAP: return REGION_NORM_MAP[stripped.title()]
    if r.title() in VALID_REGIONS:        return r.title()
    return None

STATIC_CITY_REGION = {
    "accra": "Greater Accra", "tema": "Greater Accra", "ashaiman": "Greater Accra",
    "madina": "Greater Accra", "east legon": "Greater Accra", "north legon": "Greater Accra",
    "cantonments": "Greater Accra", "dansoman": "Greater Accra", "achimota": "Greater Accra",
    "lapaz": "Greater Accra", "spintex": "Greater Accra", "osu": "Greater Accra",
    "adenta": "Greater Accra", "adenta municipality": "Greater Accra",
    "teshie": "Greater Accra", "nungua": "Greater Accra", "adabraka": "Greater Accra",
    "jamestown": "Greater Accra", "labadi": "Greater Accra", "kokomlemle": "Greater Accra",
    "amasaman": "Greater Accra", "kwabenya": "Greater Accra", "dome": "Greater Accra",
    "oyarifa": "Greater Accra", "airport residential": "Greater Accra",
    "awoshie": "Greater Accra", "weija": "Greater Accra", "haatso": "Greater Accra",
    "east cantonments": "Greater Accra", "roman ridge": "Greater Accra",
    "kaneshie": "Greater Accra", "north kaneshie": "Greater Accra",
    "darkuman": "Greater Accra", "chorkor": "Greater Accra", "okponglo": "Greater Accra",
    "legon": "Greater Accra", "agbogba": "Greater Accra", "mempeasem": "Greater Accra",
    "ashale-botwe": "Greater Accra", "dzorwulu": "Greater Accra",
    "klagon": "Greater Accra", "odorkor": "Greater Accra", "pokoase": "Greater Accra",
    "pantang": "Greater Accra", "accra central": "Greater Accra",
    "kumasi": "Ashanti", "obuasi": "Ashanti", "ejisu": "Ashanti",
    "asokore": "Ashanti", "atonsu": "Ashanti", "atonsu kumasi": "Ashanti",
    "suame": "Ashanti", "bantama": "Ashanti", "nhyiaeso": "Ashanti",
    "asawase": "Ashanti", "tafo": "Ashanti", "kwadaso": "Ashanti",
    "manso nkwanta": "Ashanti", "juaben": "Ashanti", "mampong": "Ashanti",
    "nkawie": "Ashanti", "drobonso": "Ashanti", "nkenkaso": "Ashanti",
    "kaase": "Ashanti", "bekwai": "Ashanti", "agona ashanti": "Ashanti",
    "abuakwa": "Ashanti", "afamaso": "Ashanti", "nyinamponase": "Ashanti",
    "akrofrom": "Ashanti", "wamasi": "Ashanti", "tepa": "Ashanti",
    "tikrom": "Ashanti", "santasi": "Ashanti", "buokrom": "Ashanti",
    "mankranso": "Ashanti", "kokofu": "Ashanti", "kumawu": "Ashanti",
    "donyina": "Ashanti", "asamang": "Ashanti", "offinso": "Ashanti",
    "boamadumasi": "Ashanti", "jacobu": "Ashanti", "afransi": "Ashanti",
    "takoradi": "Western", "sekondi": "Western", "axim": "Western",
    "tarkwa": "Western", "half assini": "Western", "prestea": "Western",
    "bogoso": "Western", "sefwi asawinso": "Western", "sefwi essam": "Western",
    "sefwi wiawso": "Western", "enchi": "Western", "dadieso": "Western",
    "daboase": "Western", "apremdo": "Western", "aboadze": "Western",
    "kwesimintsim": "Western", "mataheko": "Western",
    "manso amenfi": "Western", "dixcove": "Western",
    "bibiani": "Western North", "sefwi bodi": "Western North",
    "sefwi": "Western North", "juaboso": "Western North",
    "anhwiaso": "Western North",
    "koforidua": "Eastern", "nkawkaw": "Eastern", "suhum": "Eastern",
    "somanya": "Eastern", "akyem oda": "Eastern", "kade": "Eastern",
    "asamankese": "Eastern", "mpraeso": "Eastern", "abetifi": "Eastern",
    "abomosu": "Eastern", "new abirim": "Eastern", "obosomase": "Eastern",
    "cape coast": "Central", "winneba": "Central", "saltpond": "Central",
    "swedru": "Central", "ajumako": "Central", "mumford": "Central",
    "assin fosu": "Central", "kasoa": "Central", "mankessim": "Central",
    "ankaful": "Central", "buduburam": "Central", "abura": "Central",
    "ayanfuri": "Central", "agona swfru": "Central", "agona swedru": "Central",
    "agona nkwanta": "Central",   # FIX: was Oti in older versions
    "cabo corso": "Central",
    "ho": "Volta", "hohoe": "Volta", "keta": "Volta", "akatsi": "Volta",
    "aflao": "Volta", "kpandu": "Volta", "jasikan": "Volta",
    "anloga": "Volta", "sogakope": "Volta", "adidome": "Volta",
    "dambai": "Oti", "nkwanta": "Oti", "yabologu": "Oti",
    "tamale": "Northern", "walewale": "Northern", "yendi": "Northern",
    "savelugu": "Northern", "tolon": "Northern", "saboba": "Northern",
    "salaga": "Savannah", "damongo": "Savannah", "bole": "Savannah",
    "nalerigu": "North East", "bimbila": "North East", "kparigu": "North East",
    "techiman": "Bono East", "nkoranza": "Bono East", "kintampo": "Bono East",
    "atebubu": "Bono East", "yeji": "Bono East",
    "sunyani": "Brong-Ahafo", "berekum": "Brong-Ahafo",
    "banda": "Brong-Ahafo", "wenchi": "Brong-Ahafo",
    "dormaa ahenkro": "Brong-Ahafo", "abesim": "Brong-Ahafo",
    "bolgatanga": "Upper East", "navrongo": "Upper East", "bawku": "Upper East",
    "zebilla": "Upper East", "sandema": "Upper East",
    "wa": "Upper West", "lawra": "Upper West", "nandom": "Upper West",
    "jirapa": "Upper West", "nadawli": "Upper West", "daffiama": "Upper West",
    "wechiau": "Upper West", "lamboya": "Upper West",
    "goaso": "Ahafo", "bechem": "Ahafo", "duayaw nkwanta": "Ahafo",
    "kukuom": "Ahafo", "kenyasi": "Ahafo", "acherensua": "Ahafo",
}

REGION_TEXT_SIGNALS = [
    ("greater accra",  "Greater Accra"), ("accra",    "Greater Accra"),
    ("ashanti",        "Ashanti"),       ("kumasi",   "Ashanti"),
    ("western region", "Western"),       ("takoradi", "Western"),
    ("eastern region", "Eastern"),       ("koforidua","Eastern"),
    ("volta region",   "Volta"),         (" ho,",     "Volta"),
    ("central region", "Central"),       ("cape coast","Central"),
    ("northern region","Northern"),      ("tamale",   "Northern"),
    ("upper east",     "Upper East"),    ("bolgatanga","Upper East"),
    ("upper west",     "Upper West"),    (" wa,",     "Upper West"),
    ("brong",          "Brong-Ahafo"),   ("sunyani",  "Brong-Ahafo"),
    ("bono east",      "Bono East"),     ("techiman", "Bono East"),
    ("ahafo",          "Ahafo"),         ("goaso",    "Ahafo"),
    ("savannah",       "Savannah"),      ("north east region","North East"),
    ("western north",  "Western North"), ("oti region","Oti"),
]

# COMMAND ----------
# MAGIC %md ## 4. Facility-Type Inference Engine

# COMMAND ----------

_FTYPE_TYPO_MAP = {
    "farmacy":  "pharmacy",
    "pharamcy": "pharmacy",
    "pharmcy":  "pharmacy",
    "hospitl":  "hospital",
    "clinc":    "clinic",
}

_FTYPE_RULES = [
    (re.compile(r'\b(?:teaching|referral|specialist|regional|district|municipal|general|primary|secondary|mission|CHAG)\s+hospital\b', re.I), 'hospital'),
    (re.compile(r'\b(?:psychiatric|military|university)\s+hospital\b', re.I), 'hospital'),
    (re.compile(r'\bhospital\b', re.I), 'hospital'),
    (re.compile(r'\bmedical\s+(?:complex|centre|center)\b', re.I), 'hospital'),
    (re.compile(r'\bhealth\s+complex\b', re.I), 'hospital'),
    (re.compile(r'\bpharmac(?:y|ies|eutical|ist)\b', re.I), 'pharmacy'),
    (re.compile(r'\bchemist\b|\bdrug\s+(?:store|shop)\b', re.I), 'pharmacy'),
    (re.compile(r'\b(?:licensed\s+)?chemical\s+(?:seller|shop|store)\b|\bOTCMS\b|\bLCS\b', re.I), 'pharmacy'),
    (re.compile(r'\bdental\b|\bdentist\b|\bdentistry\b|\borthodont\b|\boral\b', re.I), 'dentist'),
    (re.compile(r'\bgeneral\s+practitioner\b', re.I), 'doctor'),
    (re.compile(r'\bdr\.?\s*[a-z]+\b', re.I), 'doctor'),
    (re.compile(r'\bmedical\s+practice\b|\bprivate\s+practice\b|\bphysician\b', re.I), 'doctor'),
    (re.compile(r'\bclinic\b|\bpolyclinic\b', re.I), 'clinic'),
    (re.compile(r'\bchps\b|\bcommunity\s+health\s+(?:post|center|centre)\b', re.I), 'clinic'),
    (re.compile(r'\bhealth\s+cent(?:er|re)\b', re.I), 'clinic'),
    (re.compile(r'\bhealth\s+post\b|\bhealth\s+facilit(?:y|ies)\b', re.I), 'clinic'),
    (re.compile(r'\bmaternity\s+(?:home|clinic|unit)\b', re.I), 'clinic'),
    (re.compile(r'\bdiagnostic\s+cent(?:er|re)\b', re.I), 'clinic'),
    (re.compile(r'\blaborator(?:y|ies)\b|\bmedical\s+lab\b', re.I), 'clinic'),
    (re.compile(r'\beye\s+(?:care|clinic|centre|center)\b|\boptical\b', re.I), 'clinic'),
    (re.compile(r'\bimaging\s+cent(?:er|re)\b|\bradiology\b|\bscan\b', re.I), 'clinic'),
]

def _infer_facility_type(existing, name, description, capability, org_type):
    raw = str(existing or "").strip().lower()
    if raw and raw not in ("null", "none", ""):
        corrected = _FTYPE_TYPO_MAP.get(raw, raw)
        if corrected in {"hospital", "clinic", "pharmacy", "dentist", "doctor"}:
            return corrected
    if str(org_type or "").strip().lower() == "ngo":
        return None
    for text in [name, description, capability]:
        if not text or str(text).strip().lower() in ("null", "none", ""):
            continue
        for pattern, ftype in _FTYPE_RULES:
            if pattern.search(str(text)):
                return ftype
    if str(org_type or "").strip().lower() == "facility":
        return "clinic"
    return None

@pandas_udf(StringType())
def infer_facility_type_udf(ex, nm, desc, cap, ot):
    return pd.Series([
        _infer_facility_type(e, n, d, c, o)
        for e, n, d, c, o in zip(ex, nm, desc, cap, ot)
    ])

# COMMAND ----------
# MAGIC %md ## 5. Numeric Extraction — with 24/7 False-Positive Guard

# COMMAND ----------

# SIL-V5-1: Hard guard tokens — if any of these appear near the numeric match,
# the match is REJECTED. Prevents "24/7" or "Open 24 hours" from becoming doctor_count=24.
_TIME_SOCIAL_GUARD_RE = re.compile(
    r'(?:'
    r'24\s*/\s*7'            # 24/7
    r'|24\s*hours?'          # 24 hours
    r'|\bhours?\b'           # hours
    r'|\bdays?\s+a\s+week\b' # days a week
    r'|\bopen\s+\d'          # open 24...
    r'|\blikes?\b'           # social likes
    r'|\bfollowers?\b'       # social followers
    r'|\bcheck.?ins?\b'      # checkins
    r'|\brating\b'           # rating score
    r'|\breviews?\b'         # review count
    r'|\bstars?\b'           # star rating
    r'|\bsince\s+\d{4}\b'   # since 1997
    r'|\bin\s+\d{4}\b'      # in 2015
    r')',
    re.IGNORECASE
)

def _has_time_social_context(text: str, match_start: int, window: int = 60) -> bool:
    """Check if a numeric match is near a time/social context (reject these)."""
    start = max(0, match_start - window)
    end   = min(len(text), match_start + window)
    snippet = text[start:end]
    return bool(_TIME_SOCIAL_GUARD_RE.search(snippet))

_BED_PATS = [
    re.compile(r'\bbed\s+capacity\s*[:\-=]\s*(\d[\d,]*)', re.I),
    re.compile(r'\bcapacity\s*[:\-=]\s*(\d[\d,]*)\s*(?:beds?)?', re.I),
    re.compile(r'total\s+bed\s+capacity\s*(?:of|is|:)?\s*(\d[\d,]*)', re.I),
    re.compile(r'(?:total|overall|current)\s*(?:bed|inpatient)\s*(?:count|strength|size)\s*[:\-=]?\s*(\d[\d,]*)', re.I),
    re.compile(r'(\d{1,4})\s*[-–]\s*(\d{1,4})\s*beds?', re.I),
    re.compile(r'(\d[\d,]*)\s+to\s+(\d[\d,]*)\s*beds?', re.I),
    re.compile(r'(?:about|around|approx(?:imately)?|roughly|nearly|over|more\s+than|up\s+to|at\s+least)\s*(\d[\d,]*)\s*beds?', re.I),
    re.compile(r'(?:has|with|equipped\s+with|offers|provides|features|contains|accommodates|boasts)\s*(\d[\d,]*)\s*beds?', re.I),
    re.compile(r'(\d[\d,]*)\+?\s*[-\s]?(?:bed|bedded)', re.I),
    re.compile(r'(\d[\d,]*)\s*beds?\b', re.I),
    re.compile(r'(\d[\d,]*)\s*(?:inpatient|admission|ward)\s*beds?', re.I),
    re.compile(r'bed\s+strength\s*(?:of|is|:)?\s*(\d+)', re.I),
    re.compile(r'(\d+)\s*[- ]?bed\s+(?:facility|hospital|centre|clinic)', re.I),
    re.compile(r'number\s+of\s+patient\s+beds?\s*[:\-=]?\s*(\d+)', re.I),
    re.compile(r'(\d+)\s*beds?\b', re.I),
    re.compile(r'bed\s+capacity\s*(?:of|is|:)?\s*(\d+)', re.I),
    re.compile(r'capacity\s*(?:of|is|:)?\s*(\d+)\s*(?:beds?)?', re.I),
    re.compile(r'(\d+)\s*[-–]\s*(\d+)\s*beds?', re.I),
    re.compile(r'\b(\d{1,5})\s*(?:beds?|capacity)?', re.I)
]


_DOC_PATS = [
    re.compile(r'(\d+)\s+permanent\s+doctors?\b', re.I),
    re.compile(r'(?:has|employs|staffed\s+by|team\s+of|consists\s+of)\s*(\d+)\s+(?:permanent|full-time|part-time|resident)?\s*(?:medical\s+)?(?:doctors?|physicians?|specialists?|consultants?|surgeons?|clinicians?|medical\s+officers?|house\s+officers?|general\s+practitioners?)', re.I),
    re.compile(r'(\d+)\s+(?:medical\s+officers?|specialist\s+doctors?|consultant\s+doctors?|general\s+practitioners?|resident\s+doctors?)', re.I),
    re.compile(r'(\d+)\s+medical\s+doctors?', re.I),
    re.compile(r'staff\s+strength\s+(?:of|is|:)?\s*(\d+)', re.I),
    re.compile(r'medical\s+staff(?:ing)?\s+strength\s+(?:of|is|:)?\s*(\d+)', re.I),
    re.compile(r'(?:medical|clinical|doctor|physician)\s+staff\s+(?:of|is|:)?\s*(\d+)', re.I),
    re.compile(r'team\s+of\s+(\d+)\s+(?:doctors?|surgeons?|physicians?|specialists?)', re.I),
    re.compile(r'has\s+a\s+team\s+of\s+(\d+)\s+doctors?', re.I),
    re.compile(r'(\d+)\s+(?:medical\s+)?doctors?\b', re.I),
    re.compile(r'(\d+)\s+physicians?\b', re.I),
    re.compile(r'(\d+)\s+specialists?\b', re.I),
    re.compile(r'(\d+)\s+consultants?\b', re.I),
    re.compile(r'(?:number|total)\s+of\s+doctors?\s*(?:is|:)?\s*(\d+)', re.I),
    re.compile(r'staffed\s+with\s+(\d+)\s+doctors?', re.I),
    re.compile(r'(?:has|employs|there\s+are|with)\s*(\d+)\s+doctors?', re.I),
    re.compile(r'doctor\s+strength\s*(?:of|is|:)?\s*(\d+)', re.I),
    re.compile(r'(\d+)\s+(?:doctors?|physicians?|consultants?|surgeons?)\b', re.I),
    re.compile(r'(?:team|staff|personnel|workforce)\s+(?:of\s+)?(\d+)', re.I),
    re.compile(r'(?:staff\s+strength|medical\s+staff)\s*(?:of|is|:)?\s*(\d+)', re.I),
    re.compile(r'(?:number|total)\s+of\s+doctors?\s*(?:is|:)?\s*(\d+)', re.I),
] 


# def _parse_cap_json(raw: Optional[str]) -> str:
#     if not raw or str(raw).strip().lower() in ("null", "[]", "none", ""):
#         return ""
#     for attempt in [str(raw), str(raw).replace('""', '"')]:
#         try:
#             items = json.loads(attempt)
#             if isinstance(items, list):
#                 return " ".join(str(x) for x in items)
#         except (json.JSONDecodeError, TypeError):
#             pass
#     return str(raw)
def _parse_cap_json(raw: Optional[str]) -> str:
    if not raw or str(raw).strip().lower() in ("null", "[]", "none", ""):
        return ""
    
    # Normalize JSON arrays or strings first
    for attempt in [str(raw), str(raw).replace('""', '"')]:
        try:
            items = json.loads(attempt)
            if isinstance(items, list):
                return " ".join(str(x) for x in items)
        except (json.JSONDecodeError, TypeError):
            pass
    
    # If raw is just a number, return as-is (e.g., "155" → "155")
    stripped = str(raw).strip()
    if re.fullmatch(r'\d+', stripped):
        return stripped
    
    # If raw contains digits but not pure JSON/array, clean and return
    # e.g., "Has a bed capacity of 155 beds" → keep it; "155 bed capacity" → keep it
    return stripped

def _extract_int(structured, description, cap_raw, patterns,
                 min_val=1, max_val=5000,
                 name=None, procedure_raw=None, equipment_raw=None,
                 context_words=None):

    # -------------------------
    # 1. Structured (highest priority)
    # -------------------------
    sv = str(structured or "").strip()
    if sv and sv.lower() not in ("", "null", "none", "0", "0.0"):
        try:
            v = int(float(sv))
            if min_val <= v <= max_val:
                return v
        except:
            pass

    # -------------------------
    # 2. Build text
    # -------------------------
    cap_text   = _parse_cap_json(cap_raw)
    proc_text  = " ".join(_safe_json_parse(procedure_raw)) if procedure_raw else ""
    equip_text = " ".join(_safe_json_parse(equipment_raw)) if equipment_raw else ""

    combined = " ".join([
        str(description or ""),
        str(cap_text or ""),
        str(name or ""),
        proc_text,
        equip_text,
    ]).lower()

    if not combined.strip():
        return None

    candidates = []

    # -------------------------
    # 3. Regex extraction
    # -------------------------
    for pat in patterns:
        for m in pat.finditer(combined):

            # ❌ reject time/social noise
            if _has_time_social_context(combined, m.start()):
                continue

            val = None
            groups = m.groups()

            try:
                if len(groups) >= 2 and groups[1]:
                    v1 = int(re.sub(r"\D", "", groups[0]))
                    v2 = int(re.sub(r"\D", "", groups[1]))
                    val = max(v1, v2)
                elif groups:
                    raw = re.sub(r"\D", "", groups[0])
                    if raw:
                        val = int(raw)
            except:
                continue

            if not val:
                continue

            if not (min_val <= val <= max_val):
                continue

            # -------------------------
            # 🔥 CONTEXT CHECK (NEW)
            # -------------------------
            if context_words:
                window = combined[max(0, m.start()-40): m.end()+40]
                if not any(w in window for w in context_words):
                    continue

            # -------------------------
            # ❌ ignore years
            # -------------------------
            if 1900 <= val <= 2030:
                continue

            # ❌ ignore phone-like
            if len(str(val)) >= 7:
                continue

            candidates.append(val)


    # -------------------------
    # 4. Fallback: any number near context
    # -------------------------
    if not candidates and context_words:
        for m in re.finditer(r"\b(\d{1,5})\b", combined):  # <-- changed from 1,4 to 1,5
            val = int(m.group(1))
            
            # ❌ ignore years
            if 1900 <= val <= 2030:
                continue
            
            # ❌ ignore phone-like
            if len(str(val)) >= 7:
                continue

            window = combined[max(0, m.start()-40): m.end()+40]
            if any(w in window for w in context_words):
                candidates.append(val)

    return max(candidates) if candidates else None

extract_capacity_udf = F.udf(
    lambda s, d, c, n, p, e: _extract_int(
        s, d, c, _BED_PATS,
        1, 5000,
        name=n, procedure_raw=p, equipment_raw=e,
        context_words=["bed", "beds", "capacity", "ward"]
    ),
    IntegerType()
)

extract_doctors_udf = F.udf(
    lambda s, d, c, n, p, e: _extract_int(
        s, d, c, _DOC_PATS,
        1, 2000,
        name=n, procedure_raw=p, equipment_raw=e,
        context_words=["doctor", "doctors", "physician", "staff", "consultant"]
    ),
    IntegerType()
)

# COMMAND ----------
# MAGIC %md ## 6. City Extraction

# COMMAND ----------

_CITY_FROM_TEXT_PAT = re.compile(r'\b([A-Za-z][A-Za-z\s\-]{2,}),\s*(?:Ghana|[A-Za-z]+ Region)', re.I)
_CITY_FROM_NAME_PAT = re.compile(r'[–\-]\s*([A-Za-z][A-Za-z\s]{2,}),\s*Ghana', re.I)

GHANA_CITIES = {
    # Greater Accra
    "accra","tema","ashaiman","madina","adenta","kasoa","dome","gbawwe","tafo",

    # Ashanti
    "kumasi","obuasi","tafo","effia kuma","ejisu","bekwai","mampong",

    # Western / Western North
    "takoradi","sekondi","tarkwa","axim","prestea","sefwi wiawso","bibiani",

    # Central
    "cape coast","winneba","mankessim","kasoa","dunkwa","swedru","effia-kuma",

    # Eastern
    "koforidua","suhum","nkawkaw","aburi","akosombo",

    # Volta / Oti
    "ho","hohoe","keta","aflao","dambai","nkwanta",

    # Northern / Savannah / NE
    "tamale","yendi","savelugu","walewale","damongo","nalerigu",

    # Upper East / West
    "bolgatanga","navrongo","bawku","wa","lawra","nandom",

    # Bono / Bono East / Ahafo
    "sunyani","techiman","kintampo","atebubu","goaso","bechem",

    # Additional from your data (VERY IMPORTANT)
    "boankra","domaa ahenkro","medina estates","kintampo","atebubu",
    "dome","gbawwe","effia kuma","shama","gumani","kanvile",
    "agbosome","damongo","dambai","goaso","sefwi wiawso"
}

# def _extract_city(city_field, address_line1, name):

#     # 1. direct field
#     city = normalize_text(city_field)
#     if city and city not in ("null","none",""):
#         return city

#     # 2. address_line1 lookup
#     addr = normalize_text(address_line1)
#     for c in GHANA_CITIES:
#         if c in addr:
#             return c

#     # 3. token fallback
#     tokens = addr.split()
#     tokens = [t for t in tokens if len(t) >= 4]

#     if tokens:
#         return tokens[-1]

#     # 4. name fallback
#     nm = normalize_text(name)
#     for c in GHANA_CITIES:
#         if c in nm:
#             return c

#     return ""

def _extract_city(city_field, address_line1, name):

    city = normalize_text(city_field)
    if city:
        return city

    addr = normalize_text(address_line1)

    # 🔥 NEW: handle single-word addresses
    if addr and len(addr.split()) == 1:
        return addr

    # known cities
    for c in GHANA_CITIES:
        if c in addr:
            return c

    # 🔥 fallback: last meaningful token (not road/street)
    tokens = [t for t in addr.split() if len(t) >= 4 and t not in ("road","street","lane")]
    if tokens:
        return tokens[-1]

    # name fallback
    nm = normalize_text(name)
    for c in GHANA_CITIES:
        if c in nm:
            return c

    return ""

extract_city_udf = F.udf(lambda c, a1, nm: _extract_city(c, a1, nm), StringType())

# COMMAND ----------
# MAGIC %md ## 7. Region Resolution (4-pass)

# COMMAND ----------



def _resolve_region(state_field, city, description, capability, name, address_line1, city_lookup_json):
    lookup = json.loads(city_lookup_json)

    state_norm = _norm_region_field(state_field)
    city_l = normalize_text(city)

    # -------------------------
    # 1. Strong: state_field (validated)
    # -------------------------
    if state_norm:
        if city_l and city_l in lookup:
            if lookup[city_l] == state_norm:
                return (state_norm, "state_field_strong")
            else:
                return ("Unknown", "state_city_conflict")
        return (state_norm, "state_field")

    # -------------------------
    # 2. Exact city match
    # -------------------------
    if city_l and city_l in lookup:
        return (lookup[city_l], "city_lookup")

    # -------------------------
    # 3. Fuzzy partial (SAFE)
    # -------------------------
    if city_l and len(city_l) >= 4:
        for key, val in lookup.items():
            key_norm = normalize_text(key)

            # stricter match
            if key_norm.startswith(city_l) or city_l.startswith(key_norm):
                return (val, "city_partial_safe")

    # -------------------------
    # 4. 🔥 Address line fallback (NEW)
    # -------------------------
    
    # address fallback
    addr = normalize_text(address_line1)
    for key, val in lookup.items():
        key_norm = normalize_text(key)
        if key_norm and key_norm in addr:
            return (val, "address_match")

    # -------------------------
    # 5. Text signal (last resort)
    # -------------------------
    combined = normalize_text(
        " ".join(str(x) for x in [description, capability, name] if x)
    )

    for signal, region in REGION_TEXT_SIGNALS:
        if signal in combined:
            return (region, "text_signal")

    return ("Unknown", "fallback")


# def _resolve_region(state_field, city, description, capability, name, address_line1, city_lookup_json):
#     lookup = json.loads(city_lookup_json)

#     state_norm = _norm_region_field(state_field)
#     city_l = normalize_text(city)

#     # -------------------------
#     # 1. Strong: state_field (validated)
#     # -------------------------
#     if state_norm:
#         if city_l and city_l in lookup:
#             if lookup[city_l] == state_norm:
#                 return (state_norm, "state_field_strong")
#             else:
#                 return ("Unknown", "state_city_conflict")
#         return (state_norm, "state_field")

#     # -------------------------
#     # 2. Exact city
#     # -------------------------
#     if city_l and city_l in lookup:
#         return (lookup[city_l], "city_lookup")

#     # -------------------------
#     # 3. 🔥 FUZZY MATCH (NEW)
#     # -------------------------
#     if city_l:
#         match = get_close_matches(city_l, lookup.keys(), n=1, cutoff=0.85)
#         if match:
#             return (lookup[match[0]], "city_fuzzy")

#     # -------------------------
#     # 4. Address fallback
#     # -------------------------
#     # 🔥 direct token match from address_line1
#     addr = normalize_text(address_line1)
#     for word in addr.split():
#         if word in lookup:
#             return (lookup[word], "address_token")

#     # -------------------------
#     # 5. Text signal
#     # -------------------------
#     combined = normalize_text(
#         " ".join(str(x) for x in [description, capability, name] if x)
#     )

#     for signal, region in REGION_TEXT_SIGNALS:
#         if signal in combined:
#             return (region, "text_signal")

#     return ("Unknown", "fallback")

bronze_lookup_pd = (
    spark.table(cfg.BRONZE)
         .select("address_city", "address_stateorregion")
         .toPandas()
)
data_city_region = {}
for _, row in bronze_lookup_pd.iterrows():
    city_k = str(row["address_city"] or "").strip().lower()
    norm   = _norm_region_field(str(row["address_stateorregion"] or ""))
    if city_k and norm and city_k not in ("null", "none", ""):
        data_city_region[city_k] = norm

FULL_LOOKUP = {**STATIC_CITY_REGION, **data_city_region}
print(f"City→region lookup size: {len(FULL_LOOKUP)}")
_lookup_json = json.dumps(FULL_LOOKUP)

resolve_region_udf     = F.udf(lambda sf, cy, de, ca, nm, a1: _resolve_region(sf, cy, de, ca, nm, a1, _lookup_json)[0], StringType())
resolve_region_src_udf = F.udf(lambda sf, cy, de, ca, nm, a1: _resolve_region(sf, cy, de, ca, nm, a1, _lookup_json)[1], StringType())

# COMMAND ----------
# MAGIC %md ## 8. Org-Type Heuristic

# COMMAND ----------

def _heuristic_org_type(org_type, name, source_url):
    ot = str(org_type or "").strip().lower()
    if ot and ot not in ("", "null", "none"):
        return ot
    s = f"{name or ''} {source_url or ''}".lower()
    ngo_sigs = ["foundation", "ngo ", "relief", "mission", "charity", "trust",
                " aid ", "society", "initiative", "nonprofit", "health association",
                "chag", "health service", "international"]
    fac_sigs = ["hospital", "clinic", "health centre", "health center", "medical centre",
                "medical center", "maternity", "pharmacy", "dental", "laboratory",
                "polyclinic", "diagnostic", "imaging"]
    if any(sig in s for sig in ngo_sigs):
        return "ngo"
    if any(sig in s for sig in fac_sigs):
        return "facility"
    return "facility"

heuristic_org_type_udf = F.udf(_heuristic_org_type, StringType())




# COMMAND ----------
# MAGIC %md ## 9. Enhanced Junk Filter (SIL-V5-3)

# COMMAND ----------

# SIL-V5-3: Expanded junk patterns. Key additions:
# - "NHIS accredited" / "NHIS accreditation" — admin, not clinical capability
# - "has telephone number" / "has email address" — contact noise
# - "has a location at" — address noise
# - Price/cost patterns
# - Pure "is a [facility type]" statements without clinical content
_JUNK_PATTERNS = [
    # Contact / location metadata
    r"located\s+(?:in|at|near|opposite|behind)\b",
    r"\b(?:p\.?o\.?\s*box|post\s*office)\b",
    r"(?:opposite|opp\.|behind|next\s+to)\s+",
    r"\b(?:phone|tel(?:ephone)?|fax|email|website|http|www\.)\s*[:\-]",
    r"\btelephone\s+numbers?\s+(?:are|is)\b",
    r"\bContact\s+phone\s+number\b",
    r"phone\s+number\s+is\s+\+?\d",
    r"has\s+(?:a\s+)?(?:telephone|phone)\s+number",    # SIL-V5-3 NEW
    r"has\s+(?:an?\s+)?email\s+address",               # SIL-V5-3 NEW
    r"has\s+a\s+location\s+at\b",                      # SIL-V5-3 NEW
    # Ownership / meta statements
    r"\bis\s+(?:privately|government|publicly|faith[-\s]?based)\s+owned\b",
    r"^\s*Ownership\s*[:\-]",
    r"^\s*(?:Facility\s+)?[Tt]ype\s*[:\-]\s*(?:Primary|Secondary|Tertiary|Health)",
    r"^\s*Services\s*[:\-]\s*(?:General|Outpatient|Inpatient)\b",
    # Admin / accreditation statements without clinical detail
    r"\bNHIS\s+(?:accredited|accreditation|registered)\b",  # SIL-V5-3 NEW
    r"\bis\s+(?:a|an)\s+(?:primary|secondary|tertiary|district|regional|general)\s+(?:hospital|clinic|centre|facility)\b",  # SIL-V5-3 NEW
    r"^\s*\w[\w\s]+\s+is\s+(?:a|an)\s+(?:primary|secondary|government|private)\b",  # SIL-V5-3 NEW
    r"provides\s+general\s+(?:medical\s+)?services?\b",     # SIL-V5-3 NEW (too vague)
    r"provides\s+general\s+health\s+services?\b",           # SIL-V5-3 NEW
    # Social media / engagement signals
    r"\d+\s+(?:likes|followers|check.?ins|reviews)\b",
    r"(?:always\s+open|monday|tuesday|wednesday|thursday|friday|saturday|sunday)\b",
    r"(?:facebook|instagram|twitter|whatsapp)\b",
    r"price\s+range\s+\$",                                  # SIL-V5-3 NEW
    r"(?:accepts|takes)\s+(?:cash|card|insurance)\s+only",  # SIL-V5-3 NEW
    # Page metadata
    r"page\s+(?:created|published|added|identified|transparency)",
    r"^located\b",
    r"\b(?:gps\s+address|digital\s+address|ghana\s+post\s+gps)\b",
    # LLM hallucination artifacts
    r"wait\s*[-–]\s*we\s+should\s+not",
    r"we\s+should\s+not\s+make\s+statements",
    r"is\s+a\s+Primary\s+Hospital\?\s+Wait",
    r"\bI\s+(?:cannot|can't|don't)\s+(?:provide|make|confirm)\b",
    # Open / operating hours (hours info only)
    r"^open\s+(?:24|status)",
    r"\boperating\s+hours\b",
    r"located\s+(?:in|at|near|opposite|behind)\b",
    r"\b(?:p\.?o\.?\s*box|post\s*office)\b",
    r"(?:opposite|opp\.|behind|next\s+to)\s+",
    r"\b(?:phone|tel(?:ephone)?|fax|email|website|http|www\.)\s*[:\-]?",
    r"\btelephone\s+numbers?\s+(?:are|is)\b",
    r"\bcontact\s+phone\s+number\b",
    r"phone\s+number\s+is\s+\+?\d",
    r"\bis\s+(?:privately|government|publicly|faith[-\s]?based)\s+owned\b",
    r"^\s*ownership\s*[:\-]",
    r"^\s*(?:facility\s+)?type\s*[:\-]",
    r"^\s*services\s*[:\-]\s*(?:general|outpatient|inpatient)\b",
    r"\bnhis\s+accredited\b",
    r"\bpage\s+(?:created|published|added|identified|transparency)\b",
    r"\d+\s+(?:likes|followers|check.?ins|reviews)\b",
    r"\b(?:facebook|instagram|twitter|whatsapp)\b",
    r"wait\s*[-–]\s*we\s+should\s+not",
    r"we\s+should\s+not\s+make\s+statements",
     r"^\s*page\s+(?:status|created|published|added|transparency)\b",
    r"^\s*\d+\s+(?:likes|followers|check[- ]?ins)\b",
    r"^\s*(?:open|closed)\s+(?:status|now)\b",

    # --- CONTACT ---
    r"\bwebsite\s*:",
    r"\bemail\s*:",
    r"\bphone\s*:",
    r"\bcontact\s+(?:number|details?)\b",
    r"\b(?:tel|telephone)\b\s*:",
    r"\+\d{6,}",

    # --- SOCIAL ---
    r"\b(?:likes|followers|check[- ]?ins|reviews)\b",
    r"\bfacebook|instagram|twitter|whatsapp\b",

    # --- HOURS ---
    r"\b24\s*/\s*7\b",
    r"\bopen\s+\d+\s*hours?\b",
    r"\bhours?\b",

    # --- ADMIN / NOISE ---
    r"\bnhis\b",
    r"\baccredited\b",
    r"\bprice\s+range\b",
    r"\bsince\s+\d{4}\b",

    # --- LOCATION ---
    r"\blocated\s+(?:at|in|near|opposite|behind)\b",
    r"\bp\.?o\.?\s*box\b",
]
_JUNK_RE = re.compile("|".join(_JUNK_PATTERNS), re.IGNORECASE | re.MULTILINE)
def _clean_capability_text(text):
    if not text:
        return text
    text = str(text)

    # remove contact/meta lines
    text = re.sub(r"(phone|email|website|contact).*", "", text, flags=re.I)
    text = re.sub(r"\+\d[\d\s\-]{6,}", "", text)

    return text


def _filter_clinical_array(items):
    if not items:
        return []
    out = []
    for item in items:
        s = str(item).strip()
        if len(s) < 8:
            continue
        if _JUNK_RE.search(s):
            continue
        out.append(s)
    return out

filter_clinical_udf = F.udf(_filter_clinical_array, ArrayType(StringType()))

# COMMAND ----------
# MAGIC %md ## 10. Procedure / Equipment Mining from Description (SIL-V5-4)

# COMMAND ----------

# SIL-V5-4: Secondary extraction pass — mine procedures/equipment from description
# when the structured parsed arrays are empty. Uses verb triggers for procedures,
# noun patterns for equipment.

_PROC_VERB_RE = re.compile(
    r'\b(?:performs?|offers?|provides?|conducts?|treats?|manages?|delivers?|carries\s+out|specializes\s+in|screens\s+for|diagnoses)\b'
    r'.{0,80}?'
    r'\b([a-z][a-z\s\-/]{5,60}(?:surgery|procedure|treatment|delivery|scan|test|therapy|screening|operation|care|transplant|dialysis|biopsy|resection|endoscopy|laparoscopy|c-section|vaccination|immunization|delivery))\b',
    re.IGNORECASE
)

_EQUIP_NOUN_RE = re.compile(
    r'\b((?:siemens|philips|ge|samsung|toshiba|fujifilm|mindray|drager|medtronic|olympus)\s+[a-z\s\d]{3,30}(?=\s+(?:ct|mri|scanner|machine|system))|\b(?:mri|ct\s+scan|x-ray|ultrasound|mammography|fluoroscopy|pet\s+scan|doppler|echocardiograph|endoscope|laparoscope|c-arm|ventilator|dialysis\s+machine|oxygen\s+plant|incubator|autoclave|defibrillator|infusion\s+pump|patient\s+monitor|ecg\s+machine|eeg\s+machine|laboratory\s+analyser|hematology\s+analyser|biochemistry\s+analyser|operating\s+microscope|surgical\s+robot|colposcope|slit\s+lamp)\b)',
    re.IGNORECASE
)

def _mine_procedures_from_text(description: Optional[str], existing_procs: List[str]) -> List[str]:
    """Mine clinical procedures from description when existing_procs is empty."""
    if existing_procs and len(existing_procs) > 0:
        return existing_procs
    if not description or len(str(description).strip()) < 30:
        return []
    mined = []
    seen  = set()
    for m in _PROC_VERB_RE.finditer(str(description)):
        phrase = m.group(1).strip().lower()
        if phrase not in seen and 8 <= len(phrase) <= 80:
            seen.add(phrase)
            mined.append(phrase.title())
    return mined[:10]  # cap at 10 mined items

def _mine_equipment_from_text(description: Optional[str], existing_equip: List[str]) -> List[str]:
    """Mine equipment mentions from description when existing_equip is empty."""
    if existing_equip and len(existing_equip) > 0:
        return existing_equip
    if not description or len(str(description).strip()) < 30:
        return []
    mined = []
    seen  = set()
    for m in _EQUIP_NOUN_RE.finditer(str(description)):
        phrase = m.group(0).strip().lower()
        if phrase not in seen and 3 <= len(phrase) <= 80:
            seen.add(phrase)
            mined.append(phrase.title())
    return mined[:10]

mine_procedures_udf = F.udf(_mine_procedures_from_text, ArrayType(StringType()))
mine_equipment_udf  = F.udf(_mine_equipment_from_text,  ArrayType(StringType()))

# COMMAND ----------
# MAGIC %md ## 11. Row-Level Citation Builder (SIL-V5-5)

# COMMAND ----------

# SIL-V5-5: Structured citation columns — provenance for each extracted clinical fact.
# Each citation records: field, snippet, source_column, extraction_method, confidence.
citation_item_schema = StructType([
    StructField("field",             StringType(), True),
    StructField("snippet",           StringType(), True),
    StructField("source_column",     StringType(), True),
    StructField("extraction_method", StringType(), True),
    StructField("confidence",        FloatType(),  True),
])
citations_schema = ArrayType(citation_item_schema)

def _build_citations(proc, equip, cap, spec, desc, uid):
    """Build row-level citation objects for each clinical fact type."""
    citations = []

    def make(field, items, source_col, method, conf):
        for item in (items or [])[:3]:
            s = str(item).strip()
            if len(s) >= 8:
                citations.append({
                    "field":             field,
                    "snippet":           s[:300],
                    "source_column":     source_col,
                    "extraction_method": method,
                    "confidence":        conf,
                })

    make("procedure",   proc,  "procedure",   "json_parse",  0.9)
    make("equipment",   equip, "equipment",   "json_parse",  0.9)
    make("capability",  cap,   "capability",  "json_parse",  0.85)
    make("specialty",   spec,  "specialties", "json_parse",  0.95)

    if not citations and desc and len(str(desc).strip()) >= 30:
        citations.append({
            "field":             "description",
            "snippet":           str(desc)[:300],
            "source_column":     "description",
            "extraction_method": "description_fallback",
            "confidence":        0.6,
        })
    return citations

build_citations_udf = F.udf(_build_citations, citations_schema)

# COMMAND ----------
# MAGIC %md ## 12. Apply All Transformations

# COMMAND ----------

ARRAY_COLS = [
    "specialties", "procedure", "equipment", "capability",
    "phone_numbers", "affiliationtypeids", "countries", "websites",
]

silver = bronze

silver = silver.withColumnRenamed("address_stateorregion", "address_stateOrRegion") \
               .withColumnRenamed("facilitytypeid", "facilityTypeId") \
               .withColumnRenamed("numberdoctors", "numberDoctors") \
               .withColumnRenamed("officialwebsite", "officialWebsite") \
               .withColumnRenamed("operatortypeid", "operatorTypeId")

# 12a. Parse JSON array columns
for col_name in ARRAY_COLS:
    matched = next((c for c in silver.columns if c.lower() == col_name.lower()), None)
    if matched:
        silver = silver.withColumn(f"{col_name}_parsed", safe_json_udf(F.col(matched)))

clean_capability_udf = F.udf(_clean_capability_text, StringType())

silver = silver.withColumn(
    "capability",
    clean_capability_udf(F.col("capability"))
).withColumn(
    "capability_parsed",
    filter_clinical_udf(
        F.expr("transform(capability_parsed, x -> trim(x))")
    )
)

print("✅ Array columns parsed")


def ensure_columns(df, cols):
    for col in cols:
        if col not in df.columns:
            df = df.withColumn(col, F.array().cast(ArrayType(StringType())))
    return df

silver = ensure_columns(silver, [
    "procedure_parsed",
    "equipment_parsed",
    "capability_parsed",
    "specialties_parsed"
])

# COMMAND ----------

# Specialty parent-mapping + validation
SPECIALTY_PARENT_MAP = {
    "obstetricsAndMaternityCare":               "gynecologyAndObstetrics",
    "familyPlanningAndComplexContraception":     "gynecologyAndObstetrics",
    "reproductiveEndocrinologyAndInfertility":   "gynecologyAndObstetrics",
    "maternalAndChildHealth":                    "pediatrics",
    "gynecologicalOncology":                     "medicalOncology",
    "gynecologicOncology":                       "medicalOncology",
    "clinicalPathology":                         "pathology",
    "forensicPathology":                         "pathology",
    "nuclearMedicineAndMolecularImaging":        "radiology",
    "diagnosticRadiology":                       "radiology",
    "neurosurgery":                              "generalSurgery",
    "hepatopancreatobiliarySurgery":             "generalSurgery",
    "urologicOncology":                          "urology",
    "cataractAndAnteriorSegmentSurgery":         "ophthalmology",
    "glaucomaOphthalmology":                     "ophthalmology",
    "retinaAndVitreoretinalOphthalmology":       "ophthalmology",
    "oculoplasticsAndReconstructiveOrbitalSurgery":"ophthalmology",
    "eyeTraumaAndEmergencyEyeCare":              "ophthalmology",
    "corneaOphthalmology":                       "ophthalmology",
    "refractiveSurgeryOphthalmology":            "ophthalmology",
    "aestheticDentistry":                        "dentistry",
    "generalDentistry":                          "dentistry",
    "prosthodontics":                            "dentistry",
    "orthodontics":                              "dentistry",
    "endodontics":                               "dentistry",
    "cosmeticDentistry":                         "dentistry",
    "oralAndMaxillofacialSurgery":               "dentistry",
    "periodontics":                              "dentistry",
    "pediatricDentistry":                        "dentistry",
    "pediatricEndocrinology":                    "endocrinologyAndDiabetesAndMetabolism",
    "adolescentMedicine":                        "pediatrics",
    "pediatricNeurodevelopmentalDisabilities":   "pediatrics",
    "sportsMedicinePMR":                         "physicalMedicineAndRehabilitation",
    "painMedicinePMR":                           "physicalMedicineAndRehabilitation",
    "occupationalAndEnvironmentalMedicine":      "publicHealth",
    "chronicDiseasePreventionAndLifestyleMedicine":"publicHealth",
    "preventiveMedicine":                        "publicHealth",
    "hematology":                                "medicalOncology",
    "anesthesiology":                            "anesthesia",
    "cosmeticDermatology":                       "dermatology",
}

VALID_SPECIALTIES_SET = {
    "internalMedicine","familyMedicine","pediatrics","cardiology","generalSurgery",
    "emergencyMedicine","gynecologyAndObstetrics","orthopedicSurgery","dentistry",
    "ophthalmology","radiology","pathology","infectiousDiseases","nephrology",
    "criticalCareMedicine","cardiacSurgery","plasticSurgery","neurology","psychiatry",
    "anesthesia","dermatology","urology","gastroenterology","pulmonology",
    "endocrinologyAndDiabetesAndMetabolism","neonatologyPerinatalMedicine",
    "medicalOncology","physicalMedicineAndRehabilitation","otolaryngology",
    "geriatricsInternalMedicine","hospiceAndPalliativeInternalMedicine",
    "publicHealth","globalHealthAndInternationalHealth",
    "maternalFetalMedicineOrPerinatology","socialAndBehavioralSciences",
}

def _validate_specialties(specs):
    if not specs:
        return []
    out, seen = [], set()
    for s in specs:
        if not s:
            continue
        canonical = SPECIALTY_PARENT_MAP.get(s, s)
        if canonical in VALID_SPECIALTIES_SET and canonical not in seen:
            out.append(canonical)
            seen.add(canonical)
    return out

validate_specialties_udf = F.udf(_validate_specialties, ArrayType(StringType()))
silver = silver.withColumn("specialties_parsed", validate_specialties_udf(F.col("specialties_parsed")))
print("✅ Specialty validation + parent mapping applied")

# COMMAND ----------

# Phone normalisation
def _normalize_phone(phone):
    if phone is None:
        return None
    s = str(phone).strip()
    if s.lower() in ("", "null", "none", "nan"):
        return None
    if s.startswith("+"):
        digits = "+" + re.sub(r"\D", "", s[1:])
        return digits if len(digits) > 1 else None
    digits = re.sub(r"\D", "", s)
    if len(digits) == 10 and digits.startswith("0"):
        return "+233" + digits[1:]
    if len(digits) == 9:
        return "+233" + digits
    if len(digits) == 12 and digits.startswith("233"):
        return "+" + digits
    return None

def _normalize_phone_list(raw):
    if raw is None:
        return []
    items = raw if isinstance(raw, list) else _safe_json_parse(raw)
    out = []
    for p in items:
        n = _normalize_phone(p)
        if n:
            out.append(n)
    return list(dict.fromkeys(out))

normalize_phone_list_udf = F.udf(_normalize_phone_list, ArrayType(StringType()))
normalize_country_udf     = F.udf(lambda c: "Ghana" if str(c or "").strip().lower() in ("gh", "ghana") else str(c or "").strip().title() or None, StringType())
normalize_country_code_udf = F.udf(
    lambda country, code: (str(code).strip().upper() if code and str(code).strip() else None)
                           or ("GH" if str(country or "").strip() == "Ghana" else None),
    StringType()
)

silver = (
    silver
    .withColumn("phone_numbers_parsed", normalize_phone_list_udf(F.col("phone_numbers_parsed")))
    .withColumn("official_phone",
                F.when(F.size(F.col("phone_numbers_parsed")) > 0,
                       F.element_at(F.col("phone_numbers_parsed"), 1)))
    .withColumn("address_country", normalize_country_udf(F.col("address_country")))
    .withColumn("address_countrycode",
                normalize_country_code_udf(F.col("address_country"), F.col("address_countrycode")))
)

# COMMAND ----------

def safe_int_expr(col_name, pattern=r"(\d+)"):
    return F.when(
        F.col(col_name).isNull() | (F.regexp_extract(F.col(col_name), pattern, 1) == ""),
        F.lit(None).cast(IntegerType())
    ).otherwise(F.regexp_extract(F.col(col_name), pattern, 1).cast(IntegerType()))

silver = (
    silver
    .withColumn("area_int",             safe_int_expr("area"))
    .withColumn("year_established_int", safe_int_expr("yearEstablished", r"(\d{4})"))
    .withColumn("accepts_volunteers_bool",
        F.when(F.lower(F.col("acceptsvolunteers")).isin("true", "1", "yes"), True)
        .when(F.lower(F.col("acceptsvolunteers")).isin("false", "0", "no"), False)
        .otherwise(F.lit(False)))
    .withColumn("pk_unique_id_int",
        F.when(F.col("pk_unique_id").isNull() | (F.trim(F.col("pk_unique_id")) == ""),
               F.lit(None).cast(IntegerType()))
        .otherwise(F.regexp_extract(
            F.regexp_replace(F.col("pk_unique_id"), r"\.0$", ""),
            r"(\d+)", 1
        ).cast(IntegerType())))
)

# 12c. Facility type + confidence (SIL-V5-8)
silver = silver.withColumn(
    "facility_type_clean",
    infer_facility_type_udf(
        F.col("facilitytypeid"), F.col("name"),
        F.col("description"), F.col("capability"),
        F.col("organization_type"),
    )
).withColumn(
    "facility_type_confidence",
    F.when(
        F.col("facilitytypeid").isNotNull() &
        ~F.col("facilitytypeid").isin("", "null", "None"), F.lit(1.0)
    ).when(F.col("facility_type_clean").isNotNull(), F.lit(0.8))
    .otherwise(F.lit(0.0)).cast(FloatType())
)

# 🔥 FIX: fallback inference for missing facility types
silver = silver.withColumn(
    "facility_type_clean",
    F.when(
        F.col("facility_type_clean").isNull(),
        F.when(F.col("name").rlike("(?i)hospital"), "hospital")
         .when(F.col("name").rlike("(?i)clinic"), "clinic")
         .when(F.col("name").rlike("(?i)pharmacy"), "pharmacy")
         .when(F.col("name").rlike("(?i)dental"), "dentist")
         .otherwise("unknown")
    ).otherwise(F.col("facility_type_clean"))
)

# 🔥 adjust confidence for fallback rows
silver = silver.withColumn(
    "facility_type_confidence",
    F.when(
        (F.col("facility_type_confidence") == 0.0) &
        (F.col("facility_type_clean") != "unknown"),
        F.lit(0.6)
    ).otherwise(F.col("facility_type_confidence"))
)

# 12d. Org type
silver = silver.withColumn(
    "organization_type_clean",
    heuristic_org_type_udf(F.col("organization_type"), F.col("name"), F.col("source_url"))
)

# 12e. City
silver = silver.withColumn(
    "city_clean",
    extract_city_udf(F.col("address_city"), F.col("address_line1"), F.col("name"))
)

print("✅ Structural transformations done")

# COMMAND ----------
# MAGIC %md ## 13. Junk Filter + Capacity/Doctor Extraction

# COMMAND ----------

silver = silver \
    .withColumn("capability_parsed",  filter_clinical_udf(F.col("capability_parsed"))) \
    .withColumn("procedure_parsed",   filter_clinical_udf(F.col("procedure_parsed"))) \
    .withColumn("equipment_parsed",   filter_clinical_udf(F.col("equipment_parsed")))

# SIL-V5-4: Mine from description when arrays empty
silver = silver \
    .withColumn("procedure_parsed",
                mine_procedures_udf(F.col("description"), F.col("procedure_parsed"))) \
    .withColumn("equipment_parsed",
                mine_equipment_udf(F.col("description"), F.col("equipment_parsed")))

# Capacity + doctor extraction (with time-guard)
silver = silver \
    .withColumn("capacity_int",
        extract_capacity_udf(
            F.col("capacity").cast(StringType()), F.col("description"),
            F.col("capability").cast(StringType()), F.col("name"),
            F.col("procedure_parsed").cast(StringType()),
            F.col("equipment_parsed").cast(StringType())
        )
    ) \
    .withColumn("number_doctors_int",
        extract_doctors_udf(
            F.col("numberdoctors").cast(StringType()), F.col("description"),
            F.col("capability").cast(StringType()), F.col("name"),
            F.col("procedure_parsed").cast(StringType()),
            F.col("equipment_parsed").cast(StringType())
        )
    )

cap_ct = silver.filter(F.col("capacity_int").isNotNull() & (F.col("capacity_int") > 0)).count()
doc_ct = silver.filter(F.col("number_doctors_int").isNotNull() & (F.col("number_doctors_int") > 0)).count()
tot    = silver.count()
print(f"capacity_int populated      : {cap_ct:,}  ({cap_ct/tot*100:.1f}%)")
print(f"number_doctors_int populated: {doc_ct:,}  ({doc_ct/tot*100:.1f}%)")

# 🔥 FIX: fallback heuristics for missing doctor/bed counts

# ✅ STRICT: no heuristic fallback → set 0 if missing

# silver = silver.withColumn(
#     "capacity_int",
#     F.when(F.col("capacity_int").isNull(), F.lit(0))
#      .otherwise(F.col("capacity_int"))
# )

# silver = silver.withColumn(
#     "number_doctors_int",
#     F.when(F.col("number_doctors_int").isNull(), F.lit(0))
#      .otherwise(F.col("number_doctors_int"))
# )

# COMMAND ----------
# MAGIC %md ## 14. Region Normalisation (4-pass) + Confidence

# COMMAND ----------

silver = silver.withColumn(
    "region_normalised",
    resolve_region_udf(
        F.col("address_stateorregion"), F.col("city_clean"),
        F.col("description"), F.col("capability").cast(StringType()),
        F.col("name"), F.col("address_line1"),
    )
).withColumn(
    "region_source",
    resolve_region_src_udf(
        F.col("address_stateorregion"), F.col("city_clean"),
        F.col("description"), F.col("capability").cast(StringType()),
        F.col("name"), F.col("address_line1"),
    )
).withColumn(
    "region_confidence",
    F.when(F.col("region_source") == "state_field_strong", F.lit(1.0))
     .when(F.col("region_source") == "state_field",        F.lit(0.95))
     .when(F.col("region_source") == "city_lookup",        F.lit(0.9))
     .when(F.col("region_source") == "city_partial_safe",  F.lit(0.8))
     .when(F.col("region_source") == "address_match",      F.lit(0.75))
     .when(F.col("region_source") == "text_signal",        F.lit(0.6))
     .when(F.col("region_source") == "state_city_conflict",F.lit(0.0))
     .otherwise(F.lit(0.0)).cast(FloatType())
)

unk_ct = silver.filter(F.col("region_normalised") == "Unknown").count()
print(f"region_normalised='Unknown': {unk_ct:,} / {tot:,}  ({unk_ct/tot*100:.1f}%)")
silver.groupBy("region_normalised").count().orderBy(F.desc("count")).show(20)

# COMMAND ----------
# MAGIC %md ## 15. Deduplication

# COMMAND ----------

_RICHNESS_SCALAR = [
    "number_doctors_int", "capacity_int", "area_int", "year_established_int",
    "address_line1", "city_clean", "email", "officialWebsite",
    "description", "facilitytypeid", "operatortypeid",
]
_RICHNESS_ARRAYS = [f"{c}_parsed" for c in ARRAY_COLS]

richness_expr = (
    sum(
        F.when(F.col(c).isNotNull() & (F.col(c).cast(StringType()) != ""), 1).otherwise(0)
        for c in _RICHNESS_SCALAR if c in silver.columns
    )
    + sum(
        F.when(F.size(F.col(c)) > 0, F.size(F.col(c))).otherwise(0)
        for c in _RICHNESS_ARRAYS if c in silver.columns
    )
)
silver = silver.withColumn("_row_richness", richness_expr)

w1 = Window.partitionBy("unique_id").orderBy(F.desc("_row_richness"), F.desc("ingested_at"))
silver = (silver.withColumn("_r1", F.row_number().over(w1))
                .filter(F.col("_r1") == 1).drop("_r1"))
print(f"After unique_id dedup: {silver.count():,}")

# silver = silver.withColumn(
#     "_name_norm", F.lower(F.trim(F.regexp_replace(F.col("name"), r"[^a-z0-9\s]", "")))
# ).withColumn(
#     "_city_norm", F.lower(F.trim(F.coalesce(F.col("city_clean"), F.lit(""))))
# )

silver = silver.withColumn(
    "_name_norm",
    F.lower(F.trim(F.regexp_replace(F.col("name"), r"[^a-z0-9]", "")))
).withColumn(
    "_city_norm",
    F.lower(F.trim(F.coalesce(F.col("city_clean"), F.lit(""))))
)

w2 = Window.partitionBy("_name_norm", "_city_norm") \
           .orderBy(F.desc("_row_richness"), F.desc("ingested_at"))
silver = (silver.withColumn("_r2", F.row_number().over(w2))
                .filter(F.col("_r2") == 1).drop("_r2", "_name_norm", "_city_norm"))
print(f"After cross-source dedup: {silver.count():,}")

# COMMAND ----------
# MAGIC %md ## 16. Geocoding + Geo Confidence (SIL-V5-8)

# COMMAND ----------




try:
    from geopy.geocoders import Nominatim
    from geopy.extra.rate_limiter import RateLimiter
    GEOPY_AVAILABLE = True
except ImportError:
    GEOPY_AVAILABLE = False
    print("⚠️ geopy not installed — run: %pip install geopy")

# -------------------------
# Ghana bbox validation
# -------------------------
_GHANA_LAT_MIN, _GHANA_LAT_MAX = 4.5, 11.5
_GHANA_LON_MIN, _GHANA_LON_MAX = -3.5, 1.5

def _valid_ghana_coords(lat, lon):
    return (_GHANA_LAT_MIN <= lat <= _GHANA_LAT_MAX and
            _GHANA_LON_MIN <= lon <= _GHANA_LON_MAX)

# -------------------------
# Load cache
# -------------------------
geo_cache = {
    # normalize_key(r["geo_key"]): (r["latitude"], r["longitude"])
    # for r in cache_rows
}
try:
    cache_rows = spark.table(cfg.GEO_CACHE).collect()
    geo_cache = {
        normalize_key(r["geo_key"]): (r["latitude"], r["longitude"])
        for r in cache_rows
    }
    geo_cache = {r["geo_key"]: (r["latitude"], r["longitude"]) for r in cache_rows}
    print(f"Geo cache loaded: {len(geo_cache)} entries")
except Exception as e:
    print(f"No geo cache → fresh start ({e})")

# -------------------------
# Build pandas subset
# -------------------------
silver_pd = silver.select(
    "unique_id", "name", "address_line1", "city_clean", "region_normalised"
).toPandas()

def normalize_key(text):
    text = str(text).lower()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def build_key(row):
    return normalize_key(f"{row['name']}, {row['city_clean']}, ghana")

silver_pd["geo_key"] = silver_pd.apply(build_key, axis=1)

# -------------------------
# Split cached vs new
# -------------------------
cached_mask = silver_pd["geo_key"].isin(geo_cache.keys())

cached_df = silver_pd[cached_mask].copy()
new_df    = silver_pd[~cached_mask].copy()

print(f"Cached rows: {len(cached_df)} | New rows: {len(new_df)}")

# -------------------------
# Apply cache directly
# -------------------------
cached_df["latitude"]  = cached_df["geo_key"].apply(lambda k: geo_cache[k][0])
cached_df["longitude"] = cached_df["geo_key"].apply(lambda k: geo_cache[k][1])
cached_df["geo_source"] = "cache_hit"

# -------------------------
# GEOPY PART (SAFE)
# -------------------------
new_cache_entries = []

if GEOPY_AVAILABLE and len(new_df) > 0:

    _raw_geo = Nominatim(user_agent="virtue-foundation-idp/1.0")

    _geocode = RateLimiter(
        _raw_geo.geocode,
        min_delay_seconds=1.2,   # IMPORTANT: avoid 429
        max_retries=1
    )

    def safe_geocode(query):
        for i in range(2):
            try:
                return _geocode(query, timeout=10)
            except Exception:
                time.sleep(2 * (i + 1))
        return None

    def normalize_key(text):
        text = str(text).lower()
        text = re.sub(r"[^\w\s]", "", text)   # remove punctuation
        text = re.sub(r"\s+", " ", text)      # collapse spaces
        return text.strip()


    def _geocode_row(row):
        city   = str(row.city_clean or "").strip()
        name   = str(row.name or "").strip()
        region = str(row.region_normalised or "").strip()

        # -------------------------
        # Query strategy (ordered)
        # -------------------------
        queries = []

        if city and region and region != "Unknown":
            queries.append((f"{city}, {region}, Ghana", "geopy_city_region"))

        if city:
            queries.append((f"{city}, Ghana", "geopy_city"))

        if name and city:
            queries.append((f"{name}, {city}, Ghana", "geopy_name_city"))

        # -------------------------
        # Try queries
        # -------------------------
        for q, source in queries:
            key = normalize_key(q)

            # ✅ Cache hit
            if key in geo_cache:
                lat, lon = geo_cache[key]
                if _valid_ghana_coords(lat, lon):
                    return lat, lon, source + "_cached"

            # ✅ API call
            loc = safe_geocode(q)
            if loc and _valid_ghana_coords(loc.latitude, loc.longitude):
                geo_cache[key] = (loc.latitude, loc.longitude)

                new_cache_entries.append({
                    "geo_key": key,
                    "latitude": loc.latitude,
                    "longitude": loc.longitude
                })

                return loc.latitude, loc.longitude, source

        # -------------------------
        # 🔥 SMART FALLBACK (FIXED)
        # -------------------------

        # ✅ 1. City-based fallback (CRITICAL FIX)
        if city:
            city_key = normalize_key(city)
            if city_key in STATIC_CITY_REGION:
                # optional: map to known centroid if you add CITY_COORDS
                return cfg.DEFAULT_LAT, cfg.DEFAULT_LON, "city_fallback"

        # ✅ 2. Region-based fallback
        if region and region != "Unknown":
            return cfg.DEFAULT_LAT, cfg.DEFAULT_LON, "region_fallback"

        # ✅ 3. Final fallback
        return cfg.DEFAULT_LAT, cfg.DEFAULT_LON, "country_fallback"
    # -------------------------
    # Parallel execution
    # -------------------------
    rows = list(new_df.itertuples(index=False))

    def process_row(row):
        return _geocode_row(row)

    results = []
    for i, row in enumerate(rows):
        if i % 10 == 0:
            print(f"Processing {i}/{len(rows)}")

        results.append(process_row(row))
        time.sleep(1.2)   # 🚨 CRITICAL: enforce global rate limit

    lats, lons, srcs = zip(*results)

    new_df["latitude"]   = lats
    new_df["longitude"]  = lons
    new_df["geo_source"] = srcs

else:
    # fallback if geopy unavailable
    new_df["latitude"]   = cfg.DEFAULT_LAT
    new_df["longitude"]  = cfg.DEFAULT_LON
    new_df["geo_source"] = "fallback"

# -------------------------
# Merge back
# -------------------------
final_pd = pd.concat([cached_df, new_df], ignore_index=True)

# -------------------------
# Save cache
# -------------------------
if new_cache_entries:
    spark.createDataFrame(new_cache_entries) \
        .write.format("delta") \
        .mode("append") \
        .saveAsTable(cfg.GEO_CACHE)

    print(f"✅ Added {len(new_cache_entries)} new cache entries")

# -------------------------
# Back to Spark
# -------------------------
# Build geocode result DF with only keys + geo outputs
geo_result_pd = final_pd[["unique_id", "latitude", "longitude", "geo_source"]].copy()
geo_result_spark = spark.createDataFrame(geo_result_pd)

# Join back to full silver (preserves all existing columns)
silver = (
    silver.drop("latitude", "longitude", "geo_source")
          .join(geo_result_spark, on="unique_id", how="left")
)

# geo_confidence (same logic)
silver = silver.withColumn(
    "geo_confidence",
    F.when(F.col("geo_source").contains("name_city"), F.lit(0.9))
     .when(F.col("geo_source").contains("city"),      F.lit(0.8))
     .when(F.col("geo_source").contains("cache"),     F.lit(0.85))
     .otherwise(F.lit(0.0))
)

print("✅ Geocoding complete (optimized)")

# COMMAND ----------
# MAGIC %md ## 17. Content Flags + doc_text + Citations (SIL-V5-6, SIL-V5-7, SIL-V5-5)

# COMMAND ----------

# SIL-V5-7: Content flags
silver = silver \
    .withColumn("has_procedures",   F.when(F.size(F.col("procedure_parsed"))  > 0, True).otherwise(False)) \
    .withColumn("has_equipment",    F.when(F.size(F.col("equipment_parsed"))   > 0, True).otherwise(False)) \
    .withColumn("has_capabilities", F.when(F.size(F.col("capability_parsed")) > 0, True).otherwise(False)) \
    .withColumn("has_specialties",  F.when(F.size(F.col("specialties_parsed")) > 0, True).otherwise(False)) \
    .withColumn("has_description",
        F.when(F.col("description").isNotNull() & (F.length(F.trim(F.col("description"))) >= 30), True).otherwise(False)) \
    .withColumn("has_contact",
        F.when(
            (F.size(F.col("phone_numbers_parsed")) > 0) |
            (F.col("email").isNotNull() & (F.col("email") != "")),
            True
        ).otherwise(False)) \
    .withColumn("procedure_count",  F.size(F.col("procedure_parsed"))) \
    .withColumn("equipment_count",  F.size(F.col("equipment_parsed"))) \
    .withColumn("capability_count", F.size(F.col("capability_parsed"))) \
    .withColumn("specialty_count",  F.size(F.col("specialties_parsed")))

# SIL-V5-6: doc_text + is_rag_ready
silver = silver \
    .withColumn("doc_text",
        F.trim(F.regexp_replace(
            F.concat_ws(" | ",
                F.coalesce(F.col("description"),             F.lit("")),
                F.coalesce(F.col("organizationDescription"), F.lit("")),
                F.array_join(F.col("capability_parsed"),     " "),
                F.array_join(F.col("procedure_parsed"),      " "),
                F.array_join(F.col("equipment_parsed"),      " "),
                F.array_join(F.col("specialties_parsed"),    " "),
                F.coalesce(F.col("missionStatement"),        F.lit("")),
            ),
            r"\s{2,}", " "
        ))) \
    .withColumn("doc_text_length", F.length(F.col("doc_text"))) \
    .withColumn("is_rag_ready",    F.col("doc_text_length") >= 80)

rag_ready_ct = silver.filter(F.col("is_rag_ready")).count()
print(f"RAG-ready rows (doc_text ≥ 80 chars): {rag_ready_ct:,}")

# SIL-V5-5: Row-level citations
silver = silver.withColumn(
    "citations",
    build_citations_udf(
        F.col("procedure_parsed"), F.col("equipment_parsed"),
        F.col("capability_parsed"), F.col("specialties_parsed"),
        F.col("description"), F.col("unique_id"),
    )
)

# COMMAND ----------
# MAGIC %md ## 18. Row Quality Flags (SIL-V5-9)

# COMMAND ----------

# SIL-V5-9: Add diagnostic quality flags array for downstream governance checks
def _build_quality_flags(
    number_doctors, capacity, region, facility_type, lat, lon,
    proc_count, equip_count, cap_count, description
) -> List[str]:
    flags = []
    if not number_doctors:
        flags.append("low_doctors_coverage")
    if not capacity:
        flags.append("low_capacity_coverage")
    if region == "Unknown":
        flags.append("unknown_region")
    if not facility_type:
        flags.append("missing_facility_type")
    if lat == 7.9465 and lon == -1.0232:
        flags.append("geo_fallback_centroid")
    if int(proc_count or 0) == 0 and int(equip_count or 0) == 0:
        flags.append("no_clinical_evidence")
    if int(cap_count or 0) > 5 and int(proc_count or 0) == 0:
        flags.append("capability_inflation_risk")
    if description and len(str(description).strip()) < 30:
        flags.append("thin_description")
    return flags

quality_flags_udf = F.udf(_build_quality_flags, ArrayType(StringType()))

silver = silver.withColumn(
    "row_quality_flags",
    quality_flags_udf(
        F.col("number_doctors_int"), F.col("capacity_int"),
        F.col("region_normalised"), F.col("facility_type_clean"),
        F.col("latitude"), F.col("longitude"),
        F.col("procedure_count"), F.col("equipment_count"),
        F.col("capability_count"), F.col("description"),
    )
)
silver = silver.withColumn("quality_flag_count", F.size(F.col("row_quality_flags")))

# COMMAND ----------
# MAGIC %md ## 19. Data Completeness Score

# COMMAND ----------

COMPLETENESS_WEIGHTS = {
    "number_doctors_int"    : 1.0,
    "capacity_int"          : 1.0,
    "specialties_parsed"    : 1.0,
    "procedure_parsed"      : 1.0,
    "equipment_parsed"      : 1.0,
    "capability_parsed"     : 1.0,
    "city_clean"            : 0.5,
    "region_normalised"     : 0.5,
    "officialWebsite"       : 0.5,
    "email"                 : 0.5,
    "phone_numbers_parsed"  : 0.5,
    "address_line1"         : 0.5,
    "description"           : 1.0,
    "facility_type_clean"   : 0.5,
    "operatortypeid"        : 0.5,
    "year_established_int"  : 0.3,
    "is_rag_ready"          : 0.7,  # RAG-readiness is a strong quality signal
}
_max_score = sum(COMPLETENESS_WEIGHTS.values())

_completeness_parts = []
for f, w in COMPLETENESS_WEIGHTS.items():
    if f not in silver.columns:
        continue
    if f == "region_normalised":
        part = F.when(F.col(f).isNull() | F.col(f).isin("Unknown", ""), F.lit(0.0)).otherwise(F.lit(w))
    elif f == "is_rag_ready":
        part = F.when(F.col(f) == True, F.lit(w)).otherwise(F.lit(0.0))
    elif "parsed" in f:
        part = F.when(F.size(F.col(f)) > 0, F.lit(w)).otherwise(F.lit(0.0))
    else:
        part = F.when(
            F.col(f).isNotNull() & ~F.col(f).cast(StringType()).isin("", "null", "None", "Unknown"),
            F.lit(w)
        ).otherwise(F.lit(0.0))
    _completeness_parts.append(part)

completeness_sum = reduce(op_module.add, _completeness_parts) if _completeness_parts else F.lit(0.0)
silver = silver.withColumn(
    "data_completeness_score",
    (completeness_sum / F.lit(_max_score)).cast(FloatType())
)

avg_score = silver.agg(F.avg("data_completeness_score")).collect()[0][0]
print(f"Average data completeness score: {avg_score:.3f} ({avg_score*100:.1f}%)")

# COMMAND ----------
# MAGIC %md ## 20. Canonical Schema Projection (SIL-V5-2)

# COMMAND ----------

# SIL-V5-2: Add canonical alias columns matching exact PDF schema field names.
# Bronze/Silver internal names (lowercase) are kept for backward compatibility.
# New canonical names are added as additional columns.

silver = silver \
    .withColumn("officialWebsite",     F.col("officialWebsite")) \
    .withColumn("address_stateOrRegion", F.col("address_stateorregion")) \
    .withColumn("address_zipOrPostcode", F.col("address_ziporpostcode")) \
    .withColumn("address_countryCode",   F.col("address_countrycode")) \
    .withColumn("facilityTypeId",        F.col("facility_type_clean")) \
    .withColumn("operatorTypeId",
        F.when(F.col("operatortypeid").isNotNull(),
               F.lower(F.col("operatortypeid")))
        .otherwise(F.lit(None).cast(StringType()))
    ) \
    .withColumn("numberDoctors",         F.col("number_doctors_int")) \
    .withColumn("extraction_version",    F.lit("silver_v5"))

print("✅ Canonical schema aliases added (PDF-compliant column names)")

# COMMAND ----------
# MAGIC %md ## 21. Write Silver Delta Table

# COMMAND ----------

silver_out = silver.drop("_row_richness")

(
    silver_out
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("delta.autoOptimize.optimizeWrite", "true")
    .option("delta.autoOptimize.autoCompact",    "true")
    .saveAsTable(cfg.SILVER)
)

final = spark.table(cfg.SILVER)
print(f"\n✅ Silver table written: {cfg.SILVER}")
print(f"   Rows    : {final.count():,}")
print(f"   Columns : {len(final.columns)}")

# COMMAND ----------
# MAGIC %md ## 22. Quality Report

# COMMAND ----------

s     = spark.table(cfg.SILVER)
total = s.count()

print("=" * 65)
print(f"SILVER QUALITY REPORT  ({total:,} rows)")
print("=" * 65)

CHECK_COLS = {
    "unique_id"               : ("scalar", "Unique ID present"),
    "name"                    : ("scalar", "Name present"),
    "city_clean"              : ("scalar", "City present"),
    "region_normalised"       : ("region", "Region normalised (non-Unknown)"),
    "latitude"                : ("scalar", "Latitude present"),
    "geo_source"              : ("geo",    "Geocoded (non-fallback)"),
    "facility_type_clean"     : ("scalar", "Facility type set"),
    "organization_type_clean" : ("scalar", "Org type set"),
    "number_doctors_int"      : ("scalar", "Doctor count present"),
    "capacity_int"            : ("scalar", "Bed capacity present"),
    "description"             : ("scalar", "Description present"),
    "specialties_parsed"      : ("array",  "Specialties non-empty"),
    "capability_parsed"       : ("array",  "Capability non-empty"),
    "procedure_parsed"        : ("array",  "Procedures non-empty"),
    "equipment_parsed"        : ("array",  "Equipment non-empty"),
    "is_rag_ready"            : ("bool",   "RAG-ready (doc_text ≥ 80 chars)"),
    "citations"               : ("array",  "Citations present"),
    "data_completeness_score" : ("thresh", "Completeness score ≥ 0.3"),
}

for col_name, (check_type, label) in CHECK_COLS.items():
    if col_name not in s.columns:
        print(f"  {'MISSING':8} {label}")
        continue
    if check_type == "region":
        ct = s.filter(F.col(col_name).isNotNull() & ~F.col(col_name).isin("Unknown", "")).count()
    elif check_type == "array":
        ct = s.filter(F.size(F.col(col_name)) > 0).count()
    elif check_type == "thresh":
        ct = s.filter(F.col(col_name) >= 0.3).count()
    elif check_type == "bool":
        ct = s.filter(F.col(col_name) == True).count()
    elif check_type == "geo":
        ct = s.filter(~F.col("geo_source").isin("country_fallback")).count()
    else:
        ct = s.filter(F.col(col_name).isNotNull() & (F.col(col_name).cast(StringType()) != "")).count()
    pct = ct / total * 100
    bar = "█" * int(pct / 5)
    print(f"  {label:<48} {pct:5.1f}%  {bar}")

print()
print("Geo source distribution:")
s.groupBy("geo_source").count().orderBy(F.desc("count")).show()
print("Region distribution:")
s.groupBy("region_normalised").count().orderBy(F.desc("count")).show(20)
print("Facility type distribution:")
s.groupBy("facility_type_clean").count().orderBy(F.desc("count")).show()
print("Quality flag distribution:")
s.select(F.explode("row_quality_flags").alias("flag")).groupBy("flag").count().orderBy(F.desc("count")).show(20)

avg_c = s.agg(F.avg("data_completeness_score")).collect()[0][0]
print(f"\nAverage data completeness score: {avg_c:.3f} ({avg_c*100:.1f}%)")

# COMMAND ----------

display(
    spark.table(cfg.SILVER)
         .select(
             "unique_id", "name", "organization_type_clean",
             "facilityTypeId", "city_clean", "region_normalised",
             "latitude", "longitude", "geo_source", "geo_confidence",
             "specialties_parsed", "numberDoctors", "capacity_int",
             "has_procedures", "has_equipment", "is_rag_ready",
             "data_completeness_score", "row_quality_flags",
         )
         .orderBy(F.desc("data_completeness_score"))
         .limit(20)
)

# COMMAND ----------
# MAGIC %sql
# MAGIC SELECT * FROM virtue_foundation.ghana.silver_facilities_cleaned LIMIT 50
#####################################
# OUTPUT
#####################################

"""
Run: 2026-04-24T15:40:03.344138+00:00
Bronze rows: 987  columns: 46
City→region lookup size: 271
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/udf.py:103: UserWarning: Cannot infer the eval type from type hints. 
  warnings.warn("Cannot infer the eval type from type hints. ", UserWarning)
✅ Array columns parsed
✅ Specialty validation + parent mapping applied
✅ Structural transformations done
capacity_int populated      : 35  (3.5%)
number_doctors_int populated: 11  (1.1%)
region_normalised='Unknown': 40 / 987  (4.1%)
+-----------------+-----+
|region_normalised|count|
+-----------------+-----+
|    Greater Accra|  452|
|          Ashanti|  173|
|          Western|   83|
|            Volta|   42|
|          Unknown|   40|
|          Central|   39|
|         Northern|   32|
|      Brong-Ahafo|   31|
|          Eastern|   27|
|       Upper West|   14|
|    Western North|   13|
|        Bono East|   12|
|              Oti|    8|
|            Ahafo|    7|
|       Upper East|    6|
|         Savannah|    5|
|       North East|    3|
+-----------------+-----+

After unique_id dedup: 987
After cross-source dedup: 909
No geo cache → fresh start (name 'normalize_key' is not defined)
Cached rows: 0 | New rows: 909
Processing 0/909
Processing 10/909
Processing 20/909
Processing 30/909
Processing 40/909
Processing 50/909
Processing 60/909
Processing 70/909
Processing 80/909
Processing 90/909
Processing 100/909
Processing 110/909
Processing 120/909
Processing 130/909
Processing 140/909
Processing 150/909
Processing 160/909
Processing 170/909
Processing 180/909
Processing 190/909
Processing 200/909
Processing 210/909
Processing 220/909
Processing 230/909
Processing 240/909
Processing 250/909
Processing 260/909
Processing 270/909
Processing 280/909
Processing 290/909
Processing 300/909
Processing 310/909
Processing 320/909
Processing 330/909
Processing 340/909
Processing 350/909
Processing 360/909
Processing 370/909
Processing 380/909
Processing 390/909
Processing 400/909
Processing 410/909
Processing 420/909
Processing 430/909
Processing 440/909
Processing 450/909
Processing 460/909
Processing 470/909
Processing 480/909
Processing 490/909
Processing 500/909
Processing 510/909
Processing 520/909
Processing 530/909
Processing 540/909
Processing 550/909
Processing 560/909
Processing 570/909
Processing 580/909
Processing 590/909
Processing 600/909
Processing 610/909
Processing 620/909
Processing 630/909
Processing 640/909
Processing 650/909
Processing 660/909
Processing 670/909
Processing 680/909
Processing 690/909
Processing 700/909
Processing 710/909
Processing 720/909
Processing 730/909
Processing 740/909
Processing 750/909
Processing 760/909
Processing 770/909
Processing 780/909
Processing 790/909
Processing 800/909
Processing 810/909
Processing 820/909
Processing 830/909
Processing 840/909
Processing 850/909
Processing 860/909
Processing 870/909
Processing 880/909
Processing 890/909
Processing 900/909
/home/spark-4306b952-e689-4b27-aa0a-eb/.ipykernel/11869/command-8064144905663197-3383915736:1670: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  final_pd = pd.concat([cached_df, new_df], ignore_index=True)
✅ Added 225 new cache entries
✅ Geocoding complete (optimized)
RAG-ready rows (doc_text ≥ 80 chars): 773
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/udf.py:103: UserWarning: Cannot infer the eval type from type hints. 
  warnings.warn("Cannot infer the eval type from type hints. ", UserWarning)
Average data completeness score: 0.493 (49.3%)
✅ Canonical schema aliases added (PDF-compliant column names)

✅ Silver table written: virtue_foundation.ghana.silver_facilities_cleaned
   Rows    : 909
   Columns : 90
=================================================================
SILVER QUALITY REPORT  (909 rows)
=================================================================
  Unique ID present                                100.0%  ████████████████████
  Name present                                     100.0%  ████████████████████
  City present                                     100.0%  ████████████████████
  Region normalised (non-Unknown)                   95.6%  ███████████████████
  Latitude present                                 100.0%  ████████████████████
  Geocoded (non-fallback)                           97.4%  ███████████████████
  Facility type set                                100.0%  ████████████████████
  Org type set                                     100.0%  ████████████████████
  Doctor count present                               1.2%  
  Bed capacity present                               3.6%  
  Description present                              100.0%  ████████████████████
  Specialties non-empty                             90.1%  ██████████████████
  Capability non-empty                              73.5%  ██████████████
  Procedures non-empty                              25.9%  █████
  Equipment non-empty                               10.7%  ██
  RAG-ready (doc_text ≥ 80 chars)                   85.0%  █████████████████
  Citations present                                 98.5%  ███████████████████
  Completeness score ≥ 0.3                          89.2%  █████████████████

Geo source distribution:
+--------------------+-----+
|          geo_source|count|
+--------------------+-----+
|geopy_city_region...|  560|
|   geopy_city_region|  189|
|     region_fallback|   62|
|          geopy_city|   34|
|   geopy_city_cached|   25|
|    country_fallback|   24|
|       city_fallback|   13|
|     geopy_name_city|    2|
+--------------------+-----+

Region distribution:
+-----------------+-----+
|region_normalised|count|
+-----------------+-----+
|    Greater Accra|  409|
|          Ashanti|  159|
|          Western|   78|
|          Unknown|   40|
|          Central|   38|
|            Volta|   36|
|         Northern|   31|
|      Brong-Ahafo|   29|
|          Eastern|   26|
|       Upper West|   13|
|    Western North|   12|
|        Bono East|   11|
|              Oti|    7|
|       Upper East|    6|
|            Ahafo|    6|
|         Savannah|    5|
|       North East|    3|
+-----------------+-----+

Facility type distribution:
+-------------------+-----+
|facility_type_clean|count|
+-------------------+-----+
|           hospital|  447|
|             clinic|  369|
|            unknown|   61|
|            dentist|   19|
|           pharmacy|    7|
|             doctor|    6|
+-------------------+-----+

Quality flag distribution:
+--------------------+-----+
|                flag|count|
+--------------------+-----+
|low_doctors_coverage|  898|
|low_capacity_cove...|  876|
|no_clinical_evidence|  659|
|    thin_description|  336|
|geo_fallback_cent...|   99|
|      unknown_region|   40|
|capability_inflat...|   30|
+--------------------+-----+


Average data completeness score: 0.493 (49.3%)
"""

In [0]:
%sql
SELECT * FROM virtue_foundation.ghana.silver_facilities_cleaned LIMIT 250

In [0]:
spark.table(cfg.SILVER).groupBy("region_normalised").count().orderBy(F.desc("count")).show()

In [0]:
# # Databricks notebook source
# # MAGIC %md
# # MAGIC # 02 — Silver Layer (Final v4)
# # MAGIC
# # MAGIC **All fixes applied vs v3:**
# # MAGIC
# # MAGIC | Fix ID | Description |
# # MAGIC |--------|-------------|
# # MAGIC | SIL-1  | Geocoding replaced: Nominatim → fast in-memory city dict (instant, no rate limits) |
# # MAGIC | SIL-2  | `agona nkwanta` → "Central" (was wrong "Oti"); 12 more city corrections |
# # MAGIC | SIL-3  | Address-as-name rows: strip address from name, reclassify org type |
# # MAGIC | SIL-4  | `doc_text` + `is_rag_ready` columns added (required by RAG notebook 05) |
# # MAGIC | SIL-5  | `has_procedures`, `has_equipment`, `has_capabilities`, `has_specialties` flags added |
# # MAGIC | SIL-6  | Capability junk pre-filter: removes address/contact/meta noise at Silver level |
# # MAGIC | SIL-7  | `data_completeness_score` formula includes `is_rag_ready` as a weight |
# # MAGIC | SIL-8  | `accepts_volunteers_bool` defaults to False (no nulls propagate downstream) |
# # MAGIC | SIL-9  | Phone number normalisation applied to E164 format (per PDF schema) |
# # MAGIC | SIL-10 | CHAG / NGO rows: `facility_type_clean` = NULL for NGOs (not "clinic") |
# # MAGIC
# # MAGIC **Input:**  `virtue_foundation.ghana.bronze_facilities_raw`
# # MAGIC **Output:** `virtue_foundation.ghana.silver_facilities_cleaned`

# # COMMAND ----------
# # DBTITLE 1,Step 0 — Imports & Config

# import json
# import re
# import time
# from datetime import datetime
# from functools import reduce
# import operator as op_module
# from typing import Optional, List

# import pandas as pd
# from pyspark.sql import SparkSession, functions as F
# from pyspark.sql.types import (
#     StringType, IntegerType, FloatType, BooleanType,
#     ArrayType, DoubleType,
# )
# from pyspark.sql.window import Window

# spark = SparkSession.builder.getOrCreate()
# print(f"Run: {datetime.now().isoformat()}")

# spark.sql("CREATE DATABASE IF NOT EXISTS virtue_foundation.ghana")

# class Config:
#     BRONZE         = "virtue_foundation.ghana.bronze_facilities_raw"
#     SILVER         = "virtue_foundation.ghana.silver_facilities_cleaned"
#     DEFAULT_LAT    = 7.9465
#     DEFAULT_LON    = -1.0232

# cfg = Config()

# # COMMAND ----------
# # DBTITLE 1,Step 1 — Read Bronze

# bronze = spark.table(cfg.BRONZE)
# raw_count = bronze.count()
# print(f"Bronze rows: {raw_count:,}  columns: {len(bronze.columns)}")

# # COMMAND ----------
# # DBTITLE 1,Step 2 — JSON Array Parser UDF

# def _safe_json_parse(text: Optional[str]) -> List[str]:
#     """Parse JSON arrays that may have double-escaped quotes from CSV export."""
#     if text is None or str(text).strip() in ("", "null", "[]", "None"):
#         return []
#     text = str(text).strip()
#     for attempt in [text, text.replace('""', '"')]:
#         if attempt.startswith('"[') and attempt.endswith(']"'):
#             attempt = attempt[1:-1]
#         try:
#             result = json.loads(attempt)
#             if isinstance(result, list):
#                 return [str(x).strip() for x in result if x is not None and str(x).strip()]
#             return [str(result)]
#         except json.JSONDecodeError:
#             pass
#     # Last-resort: comma-split stripped string
#     cleaned = text.strip('"').strip("'")
#     if "," in cleaned:
#         return [x.strip().strip('"').strip("'") for x in cleaned.split(",") if x.strip()]
#     return [cleaned] if cleaned else []

# safe_json_udf = F.udf(_safe_json_parse, ArrayType(StringType()))

# # COMMAND ----------
# # DBTITLE 1,Step 3 — Address-as-Name Fix (SIL-3)
# # Rows where `name` contains an address string (scraped incorrectly from LinkedIn etc.)
# # are fixed: extract the real name from `organizationdescription` or use a fallback.

# _ADDRESS_IN_NAME = re.compile(
#     r"^\d+/|\b(road|rd\.|street|st\.|avenue|lane|highway)\b|Ghana\s*$",
#     re.I
# )

# def _fix_name(name, description, source_url):
#     """If name looks like an address, try to recover real name from description or URL."""
#     if not name or not _ADDRESS_IN_NAME.search(str(name)):
#         return str(name or "").strip() or None
#     # Try description first word(s)
#     desc = str(description or "").strip()
#     if desc and len(desc) < 80:
#         return desc.split(".")[0].strip()[:100]
#     # Try URL domain as fallback name
#     url = str(source_url or "").strip()
#     m = re.search(r"//(?:www\.)?([^/]+)", url)
#     if m:
#         domain = m.group(1).split(".")[0].replace("-", " ").title()
#         if 3 < len(domain) < 50:
#             return domain
#     return str(name or "").strip()[:100]

# fix_name_udf = F.udf(_fix_name, StringType())

# silver = bronze.withColumn(
#     "name",
#     fix_name_udf(F.col("name"), F.col("description"), F.col("source_url"))
# )

# # COMMAND ----------
# # DBTITLE 1,Step 4 — Phone Normalisation to E164 (per PDF schema)

# def _e164(phone: Optional[str]) -> Optional[str]:
#     if not phone:
#         return None
#     s = str(phone).strip()
#     if s.lower() in ("", "null", "none", "nan"):
#         return None
#     if s.startswith("+"):
#         digits = "+" + re.sub(r"\D", "", s[1:])
#         return digits if len(digits) >= 10 else None
#     digits = re.sub(r"\D", "", s)
#     if len(digits) == 10 and digits.startswith("0"):
#         return "+233" + digits[1:]
#     if len(digits) == 9:
#         return "+233" + digits
#     if len(digits) == 12 and digits.startswith("233"):
#         return "+" + digits
#     return None

# def _e164_list(raw: Optional[str]) -> List[str]:
#     items = _safe_json_parse(raw)
#     out = []
#     for p in items:
#         n = _e164(p)
#         if n:
#             out.append(n)
#     return list(dict.fromkeys(out))   # deduplicate, preserve order

# e164_list_udf = F.udf(_e164_list, ArrayType(StringType()))

# # COMMAND ----------
# # DBTITLE 1,Step 5 — Parse all JSON array columns

# ARRAY_COLS = [
#     "specialties", "procedure", "equipment", "capability",
#     "phone_numbers", "affiliationtypeids", "countries", "websites",
# ]

# for col_name in ARRAY_COLS:
#     matched = next((c for c in silver.columns if c.lower() == col_name.lower()), None)
#     if matched:
#         silver = silver.withColumn(f"{col_name}_parsed", safe_json_udf(F.col(matched)))

# # Override phone_numbers_parsed with E164-normalised version
# silver = silver.withColumn(
#     "phone_numbers_parsed",
#     e164_list_udf(F.col("phone_numbers"))
# ).withColumn(
#     "official_phone",
#     F.when(F.size(F.col("phone_numbers_parsed")) > 0,
#            F.element_at(F.col("phone_numbers_parsed"), 1))
# )

# # Country normalisation (per PDF: address_country = full name, address_countrycode = ISO alpha-2)
# silver = silver \
#     .withColumn("address_country",
#         F.when(F.lower(F.col("address_country")).isin("gh","ghana"), F.lit("Ghana"))
#          .otherwise(F.col("address_country"))) \
#     .withColumn("address_countrycode",
#         F.when(F.col("address_countrycode").isNull() &
#                (F.col("address_country") == "Ghana"), F.lit("GH"))
#          .otherwise(F.upper(F.col("address_countrycode"))))

# print("✅ JSON arrays parsed and normalised")

# # COMMAND ----------
# # DBTITLE 1,Step 6 — Capability Junk Pre-Filter (SIL-6)
# # Strip address/contact/metadata noise from capability_parsed at Silver level.
# # Downstream (IDP Agent) then works with clean clinical capabilities only.

# CAP_JUNK_PATTERNS = [
#     r"(?i)^located\s+(at|in)\b",
#     r"(?i)^p\.?\s*o\.?\s*box\s+",
#     r"(?i)\bGPS\s+address\b",
#     r"(?i)\bofficial\s+website\b",
#     r"(?i)\bfacebook\s+page\b",
#     r"(?i)^always\s+open$",
#     r"(?i)^phone\s*(number|contact)?:",
#     r"(?i)\btelephone\b",
#     r"(?i)\bemail[:\s]",
#     r"(?i)^fax[:\s]",
#     r"(?i)^\+\d{7,}",                         # raw phone number
#     r"(?i)\bnhis\s*accredited\b",              # admin note, not clinical
#     r"(?i)\bdiocesian?\s+affiliation\b",
#     r"(?i)\bregistered\s+(with|on)\b",
#     r"(?i)\bpage\s+created\b",
#     r"(?i)^(insurance|health\s+insurance)\s+packages",
#     r"https?://",                               # URLs
#     r"(?i)^(p\.?\s*o\.?\s*box|box\s+\d+)",
#     r"(?i)\boperating\s+hours\b",
#     r"(?i)^open\s+(status|24)",
#     r"(?i)\boffers\s+(online|in.store|curbside|delivery|reservation)",  # e-commerce noise
# ]

# def _prefilter_capability(raw: Optional[str]) -> List[str]:
#     items = _safe_json_parse(raw)
#     result = []
#     for item in items:
#         s = str(item).strip()
#         if not s or len(s) < 8:
#             continue
#         if not any(re.search(p, s) for p in CAP_JUNK_PATTERNS):
#             result.append(s)
#     return result

# prefilter_cap_udf = F.udf(_prefilter_capability, ArrayType(StringType()))

# silver = silver.withColumn(
#     "capability_parsed",
#     prefilter_cap_udf(F.col("capability"))
# )

# # COMMAND ----------
# # DBTITLE 1,Step 7 — Numeric Extraction from Free-Text

# _BED_PATS = [
#     re.compile(r"\bbed\s+capacity\s*[:\-=]\s*(\d[\d,]*)", re.I),
#     re.compile(r"\bcapacity\s*[:\-=]\s*(\d[\d,]*)\s*(?:beds?)?", re.I),
#     re.compile(r"(\d{1,4})\s*[-–]\s*(\d{1,4})\s*beds?", re.I),       # range → max
#     re.compile(r"(?:about|over|more\s+than)\s*(\d[\d,]*)\s*beds?", re.I),
#     re.compile(r"(\d[\d,]*)\+?\s*[-\s]?(?:bed|bedded)\s*(?:hospital|facility|centre|ward)?", re.I),
#     re.compile(r"(\d[\d,]*)\s*beds?\b", re.I),
#     re.compile(r"(\d[\d,]*)\s*(?:inpatient|admission)", re.I),
# ]

# _DOC_PATS = [
#     re.compile(r"(\d+)\s+permanent\s+doctors?\b", re.I),
#     re.compile(r"(\d+)\s+(?:medical\s+)?doctors?\b", re.I),
#     re.compile(r"(\d+)\s+physicians?\b", re.I),
#     re.compile(r"(\d+)\s+specialists?\b", re.I),
#     re.compile(r"(\d+)\s+consultants?\b", re.I),
#     re.compile(r"(?:team|staff)\s+(?:of\s+)?(\d+)\s+(?:medical|clinical)", re.I),
#     re.compile(r"medical\s+team\s+(?:consists?\s+of\s+)?(\d+)", re.I),
#     re.compile(r"(\d+)\s+clinicians?\b", re.I),
#     re.compile(r"staff\s+of\s+(\d+)", re.I),
# ]

# def _parse_cap_text(raw: Optional[str]) -> str:
#     if not raw or str(raw).strip().lower() in ("null","[]","none",""):
#         return ""
#     for attempt in [str(raw), str(raw).replace('""', '"')]:
#         try:
#             items = json.loads(attempt)
#             if isinstance(items, list):
#                 return " ".join(str(x) for x in items)
#         except Exception:
#             pass
#     return str(raw)

# def _extract_int(structured, description, cap_raw, patterns, min_val=1, max_val=5000):
#     sv = str(structured or "").strip()
#     if sv and sv not in ("", "null", "None", "0", "0.0"):
#         try:
#             v = int(float(sv))
#             if min_val <= v <= max_val:
#                 return v
#         except (ValueError, TypeError):
#             pass
#     combined = f"{description or ''} {_parse_cap_text(cap_raw)}".strip()
#     if not combined:
#         return None
#     for pat in patterns:
#         m = pat.search(combined)
#         if m:
#             # handle range patterns (2 groups → take max)
#             try:
#                 if pat.groups == 2:
#                     vals = [int(re.sub(r"[,+\s]", "", g)) for g in m.groups() if g]
#                     v = max(vals)
#                 else:
#                     v = int(re.sub(r"[,+\s]", "", m.group(1)))
#                 if min_val <= v <= max_val:
#                     return v
#             except (ValueError, IndexError):
#                 continue
#     return None

# capacity_udf = F.udf(
#     lambda s, d, c: _extract_int(s, d, c, _BED_PATS, 1, 5000),
#     IntegerType()
# )
# doctors_udf = F.udf(
#     lambda s, d, c: _extract_int(s, d, c, _DOC_PATS, 1, 2000),
#     IntegerType()
# )

# # COMMAND ----------
# # DBTITLE 1,Step 8 — Facility Type Inference Engine

# _FTYPE_TYPO_MAP = {
#     "farmacy":"pharmacy", "pharamcy":"pharmacy", "pharmcy":"pharmacy",
#     "hospitl":"hospital", "clinc":"clinic",
# }

# _FTYPE_RULES = [
#     (re.compile(r"\b(?:teaching|referral|specialist|regional|district|municipal|general)\s+hospital\b", re.I), "hospital"),
#     (re.compile(r"\b(?:psychiatric|military|university)\s+hospital\b", re.I), "hospital"),
#     (re.compile(r"\bhospital\b", re.I), "hospital"),
#     (re.compile(r"\bmedical\s+(?:complex|centre|center)\b", re.I), "hospital"),
#     (re.compile(r"\bhealth\s+complex\b", re.I), "hospital"),
#     (re.compile(r"\bpharmac(?:y|ies|eutical|ist)\b", re.I), "pharmacy"),
#     (re.compile(r"\bchemist\b|\bdrug\s+(?:store|shop)\b", re.I), "pharmacy"),
#     (re.compile(r"\bdental\b|\bdentist\b|\bdentistry\b|\borthodont\b|\boral\s+health\b", re.I), "dentist"),
#     (re.compile(r"\bgeneral\s+practitioner\b", re.I), "doctor"),
#     (re.compile(r"\bclinic\b|\bpolyclinic\b", re.I), "clinic"),
#     (re.compile(r"\bchps\b|\bcommunity\s+health\s+(?:post|center|centre)\b", re.I), "clinic"),
#     (re.compile(r"\bhealth\s+cent(?:er|re)\b", re.I), "clinic"),
#     (re.compile(r"\bhealth\s+post\b|\bhealth\s+facilit", re.I), "clinic"),
#     (re.compile(r"\bmaternity\s+(?:home|clinic|unit)\b", re.I), "clinic"),
#     (re.compile(r"\bdiagnostic\s+cent(?:er|re)\b", re.I), "clinic"),
#     (re.compile(r"\blaborator(?:y|ies)\b|\bmedical\s+lab\b", re.I), "clinic"),
#     (re.compile(r"\beye\s+(?:care|clinic|centre|center)\b|\boptical\b", re.I), "clinic"),
#     (re.compile(r"\bimaging\s+cent(?:er|re)\b", re.I), "clinic"),
# ]

# def _infer_facility_type(existing, name, description, capability, org_type):
#     raw = str(existing or "").strip().lower()
#     if raw and raw not in ("null", "none", ""):
#         corrected = _FTYPE_TYPO_MAP.get(raw, raw)
#         if corrected in {"hospital", "clinic", "pharmacy", "dentist", "doctor"}:
#             return corrected
#     # NGOs do not have a facility type
#     if str(org_type or "").strip().lower() == "ngo":
#         return None
#     for text in [name, description, capability]:
#         if not text or str(text).strip().lower() in ("null", "none", ""):
#             continue
#         for pattern, ftype in _FTYPE_RULES:
#             if pattern.search(str(text)):
#                 return ftype
#     if str(org_type or "").strip().lower() == "facility":
#         return "clinic"
#     return None

# infer_facility_type_udf = F.udf(
#     lambda ex, nm, desc, cap, ot: _infer_facility_type(ex, nm, desc, cap, ot),
#     StringType()
# )

# # COMMAND ----------
# # DBTITLE 1,Step 9 — Organisation Type Heuristic

# def _heuristic_org_type(org_type, name, source_url):
#     ot = str(org_type or "").strip().lower()
#     if ot and ot not in ("", "null", "none"):
#         return ot
#     s = f"{name or ''} {source_url or ''}".lower()
#     ngo_sigs = [
#         "foundation", "ngo ", "relief", "mission", "charity",
#         "trust", " aid ", "society", "initiative", "nonprofit",
#         "health association", "chag", "health service", "international",
#         "global health", "world health", "united nations", "unicef", "who ",
#     ]
#     fac_sigs = [
#         "hospital", "clinic", "health centre", "health center",
#         "medical centre", "medical center", "maternity",
#         "pharmacy", "dental", "laboratory", "polyclinic",
#         "diagnostic", "imaging",
#     ]
#     if any(sig in s for sig in ngo_sigs):
#         return "ngo"
#     if any(sig in s for sig in fac_sigs):
#         return "facility"
#     return "facility"

# heuristic_org_type_udf = F.udf(_heuristic_org_type, StringType())

# # COMMAND ----------
# # DBTITLE 1,Step 10 — City Extraction from address_line1 / name

# _CITY_FROM_TEXT_PAT = re.compile(
#     r"\b([A-Za-z][A-Za-z\s\-]{2,}),\s*(?:Ghana|[A-Za-z]+ Region)",
#     re.I
# )
# _CITY_FROM_NAME_PAT = re.compile(r"[–\-]\s*([A-Za-z][A-Za-z\s]{2,}),\s*Ghana", re.I)

# def _extract_city(city_field, address_line1, name):
#     raw = str(city_field or "").strip()
#     if raw and raw.lower() not in ("null", "none", ""):
#         return raw.title()
#     for text in [address_line1, name]:
#         if not text or str(text).strip().lower() in ("null", "none", ""):
#             continue
#         m = _CITY_FROM_TEXT_PAT.search(str(text))
#         if m:
#             city = m.group(1).strip()
#             if 3 <= len(city) <= 40:
#                 return city.title()
#         m2 = _CITY_FROM_NAME_PAT.search(str(text))
#         if m2:
#             city = m2.group(1).strip()
#             if 3 <= len(city) <= 40:
#                 return city.title()
#     return None

# extract_city_udf = F.udf(_extract_city, StringType())

# # COMMAND ----------
# # DBTITLE 1,Step 11 — Region Constants + City→Region Lookup (SIL-1, SIL-2)

# VALID_REGIONS = {
#     "Ashanti", "Greater Accra", "Western", "Eastern", "Central",
#     "Volta", "Northern", "Upper East", "Upper West",
#     "Oti", "Bono East", "Ahafo", "Savannah", "North East",
#     "Western North", "Brong-Ahafo",
# }

# REGION_NORM_MAP = {
#     "Ashanti Region":"Ashanti", "Ashanti":"Ashanti", "Asante":"Ashanti",
#     "Greater Accra Region":"Greater Accra", "Greater Accra":"Greater Accra", "Accra":"Greater Accra",
#     "Western Region":"Western", "Western":"Western",
#     "Eastern Region":"Eastern", "Eastern":"Eastern",
#     "Volta Region":"Volta", "Volta":"Volta",
#     "Central Region":"Central", "Central":"Central",
#     "Brong Ahafo Region":"Brong-Ahafo", "Brong-Ahafo Region":"Brong-Ahafo",
#     "Brong Ahafo":"Brong-Ahafo", "Brong-Ahafo":"Brong-Ahafo",
#     "Bono Region":"Brong-Ahafo",
#     "Northern Region":"Northern", "Northern":"Northern",
#     "Upper East Region":"Upper East", "Upper East":"Upper East",
#     "Upper West Region":"Upper West", "Upper West":"Upper West",
#     "Oti Region":"Oti", "Oti":"Oti",
#     "Bono East Region":"Bono East", "Bono East":"Bono East",
#     "Ahafo Region":"Ahafo", "Ahafo":"Ahafo",
#     "Savannah Region":"Savannah", "Savannah":"Savannah",
#     "North East Region":"North East", "North East":"North East",
#     "Western North Region":"Western North", "Western North":"Western North",
#     # Sub-district aliases
#     "Asutifi South":"Ahafo", "Asutifi":"Ahafo",
#     "Central Tongu District":"Volta", "Kumasi":"Ashanti",
#     "Tamale":"Northern", "North Legon":"Greater Accra",
#     "Ga East Municipality":"Greater Accra",
# }

# def _norm_region_field(r):
#     if not r or str(r).strip().lower() in ("", "null", "none"):
#         return None
#     r = str(r).strip()
#     if r in REGION_NORM_MAP: return REGION_NORM_MAP[r]
#     if r.title() in REGION_NORM_MAP: return REGION_NORM_MAP[r.title()]
#     stripped = re.sub(r"\s*[Rr]egion$", "", r).strip()
#     if stripped in REGION_NORM_MAP: return REGION_NORM_MAP[stripped]
#     if stripped.title() in REGION_NORM_MAP: return REGION_NORM_MAP[stripped.title()]
#     if r.title() in VALID_REGIONS: return r.title()
#     return None

# # ── FIX SIL-2: Corrected city→region mapping ─────────────────────────────────
# # Previous version had "agona nkwanta" → "Oti" which is WRONG.
# # Agona Nkwanta is in Central Region.
# STATIC_CITY_REGION = {
#     # Greater Accra
#     "accra":"Greater Accra","tema":"Greater Accra","ashaiman":"Greater Accra",
#     "madina":"Greater Accra","east legon":"Greater Accra","north legon":"Greater Accra",
#     "cantonments":"Greater Accra","dansoman":"Greater Accra","achimota":"Greater Accra",
#     "lapaz":"Greater Accra","spintex":"Greater Accra","osu":"Greater Accra",
#     "adenta":"Greater Accra","teshie":"Greater Accra","nungua":"Greater Accra",
#     "adabraka":"Greater Accra","jamestown":"Greater Accra","labadi":"Greater Accra",
#     "kokomlemle":"Greater Accra","amasaman":"Greater Accra","kwabenya":"Greater Accra",
#     "dome":"Greater Accra","oyarifa":"Greater Accra","airport residential":"Greater Accra",
#     "awoshie":"Greater Accra","weija":"Greater Accra","haatso":"Greater Accra",
#     "east cantonments":"Greater Accra","roman ridge":"Greater Accra",
#     "kaneshie":"Greater Accra","north kaneshie":"Greater Accra",
#     "darkuman":"Greater Accra","chorkor":"Greater Accra","okponglo":"Greater Accra",
#     "legon":"Greater Accra","agbogba":"Greater Accra","mempeasem":"Greater Accra",
#     "ashale-botwe":"Greater Accra","dzorwulu":"Greater Accra",
#     "klagon":"Greater Accra","odorkor":"Greater Accra","pokoase":"Greater Accra",
#     "pantang":"Greater Accra","accra central":"Greater Accra","ridge":"Greater Accra",
#     "kwabenyan":"Greater Accra","adenta municipality":"Greater Accra",
#     # Ashanti
#     "kumasi":"Ashanti","obuasi":"Ashanti","ejisu":"Ashanti",
#     "asokore":"Ashanti","atonsu":"Ashanti","atonsu kumasi":"Ashanti",
#     "suame":"Ashanti","bantama":"Ashanti","nhyiaeso":"Ashanti",
#     "asawase":"Ashanti","tafo":"Ashanti","kwadaso":"Ashanti",
#     "manso nkwanta":"Ashanti","juaben":"Ashanti","mampong":"Ashanti",
#     "nkawie":"Ashanti","drobonso":"Ashanti","nkenkaso":"Ashanti",
#     "kaase":"Ashanti","bekwai":"Ashanti","agona ashanti":"Ashanti",
#     "abuakwa":"Ashanti","afamaso":"Ashanti","nyinamponase":"Ashanti",
#     "akrofrom":"Ashanti","wamasi":"Ashanti","tepa":"Ashanti",
#     "tikrom":"Ashanti","santasi":"Ashanti","buokrom":"Ashanti",
#     "mankranso":"Ashanti","kokofu":"Ashanti","kumawu":"Ashanti",
#     "donyina":"Ashanti","asamang":"Ashanti","offinso":"Ashanti",
#     "boamadumasi":"Ashanti","jacobu":"Ashanti","afransi":"Ashanti",
#     "asuofia":"Ashanti","apenkro":"Ashanti","sromani":"Ashanti","asin":"Ashanti",
#     # Western
#     "takoradi":"Western","sekondi":"Western","axim":"Western",
#     "tarkwa":"Western","half assini":"Western","prestea":"Western",
#     "bogoso":"Western","sefwi asawinso":"Western","sefwi essam":"Western",
#     "enchi":"Western","daboase":"Western","adumkrom":"Western",
#     "adjoum":"Western","apremdo":"Western","aboadze":"Western",
#     "kwesimintsim":"Western","mataheko":"Western","manso amenfi":"Western",
#     "dixcove":"Western","adum banso":"Western","dadieso":"Western",
#     # Western North
#     "bibiani":"Western North","sefwi bodi":"Western North",
#     "sefwi wiawso":"Western North","sefwi":"Western North",
#     "juaboso":"Western North","anhwiaso":"Western North",
#     # Eastern
#     "koforidua":"Eastern","nkawkaw":"Eastern","suhum":"Eastern",
#     "somanya":"Eastern","akyem oda":"Eastern","kade":"Eastern",
#     "asamankese":"Eastern","mpraeso":"Eastern","abetifi":"Eastern",
#     "abomosu":"Eastern","new abirim":"Eastern","obosomase":"Eastern",
#     "nsuta":"Eastern",
#     # Central — FIX SIL-2: agona nkwanta is CENTRAL, not Oti
#     "cape coast":"Central","winneba":"Central","saltpond":"Central",
#     "swedru":"Central","ajumako":"Central","mumford":"Central",
#     "assin fosu":"Central","kasoa":"Central","mankessim":"Central",
#     "ankaful":"Central","buduburam":"Central","abura":"Central",
#     "agona nkwanta":"Central",   # ← CORRECTED (was "Oti" in v3)
#     "agona swfru":"Central","agona swedru":"Central",
#     "cabo corso":"Central","ayanfuri":"Central",
#     # Volta
#     "ho":"Volta","hohoe":"Volta","keta":"Volta","akatsi":"Volta",
#     "aflao":"Volta","kpandu":"Volta","jasikan":"Volta",
#     "anloga":"Volta","sogakope":"Volta","adidome":"Volta","ateiku":"Volta",
#     # Oti
#     "dambai":"Oti","nkwanta":"Oti","yabologu":"Oti",
#     # Northern
#     "tamale":"Northern","walewale":"Northern","yendi":"Northern",
#     "savelugu":"Northern","tolon":"Northern","saboba":"Northern",
#     "karaga":"Northern","nogsenia":"Northern","kawkawti":"Northern",
#     # Savannah
#     "salaga":"Savannah","damongo":"Savannah","bole":"Savannah",
#     "kabiase gonja":"Savannah",
#     # North East
#     "nalerigu":"North East","bimbila":"North East","kparigu":"North East",
#     # Bono East
#     "techiman":"Bono East","nkoranza":"Bono East","kintampo":"Bono East",
#     "atebubu":"Bono East","yeji":"Bono East",
#     # Brong-Ahafo
#     "sunyani":"Brong-Ahafo","berekum":"Brong-Ahafo","banda":"Brong-Ahafo",
#     "wenchi":"Brong-Ahafo","dormaa ahenkro":"Brong-Ahafo","abesim":"Brong-Ahafo",
#     # Upper East
#     "bolgatanga":"Upper East","navrongo":"Upper East","bawku":"Upper East",
#     "zebilla":"Upper East","sandema":"Upper East",
#     # Upper West
#     "wa":"Upper West","lawra":"Upper West","nandom":"Upper West",
#     "jirapa":"Upper West","nadawli":"Upper West","daffiama":"Upper West",
#     "wechiau":"Upper West","lamboya":"Upper West","gwo":"Upper West",
#     # Ahafo
#     "goaso":"Ahafo","bechem":"Ahafo","duayaw nkwanta":"Ahafo",
#     "kukuom":"Ahafo","kenyasi":"Ahafo","acherensua":"Ahafo","mepom":"Ahafo",
# }

# # Text signals for null-city rows
# REGION_TEXT_SIGNALS = [
#     ("greater accra","Greater Accra"),("accra","Greater Accra"),
#     ("ashanti","Ashanti"),("kumasi","Ashanti"),
#     ("western region","Western"),("takoradi","Western"),("sekondi","Western"),
#     ("eastern region","Eastern"),("koforidua","Eastern"),
#     ("volta region","Volta"),(", ho,","Volta"),
#     ("central region","Central"),("cape coast","Central"),
#     ("northern region","Northern"),("tamale","Northern"),
#     ("upper east","Upper East"),("bolgatanga","Upper East"),
#     ("upper west","Upper West"),(", wa,","Upper West"),
#     ("brong","Brong-Ahafo"),("sunyani","Brong-Ahafo"),
#     ("bono east","Bono East"),("techiman","Bono East"),
#     ("ahafo","Ahafo"),("goaso","Ahafo"),
#     ("savannah","Savannah"),("north east region","North East"),
#     ("western north","Western North"),("oti region","Oti"),
# ]

# # Build data-driven city→region lookup from bronze at runtime
# _bronze_lookup = (
#     spark.table(cfg.BRONZE)
#          .select("address_city","address_stateorregion")
#          .toPandas()
# )
# _data_city_region = {}
# for _, row in _bronze_lookup.iterrows():
#     city_k = str(row["address_city"] or "").strip().lower()
#     norm   = _norm_region_field(str(row["address_stateorregion"] or ""))
#     if city_k and norm and city_k not in ("null","none",""):
#         _data_city_region[city_k] = norm

# # Merge: static takes precedence for correctness; data-driven fills gaps
# FULL_LOOKUP = {**_data_city_region, **STATIC_CITY_REGION}
# _LOOKUP_JSON = json.dumps(FULL_LOOKUP)
# print(f"City→region lookup size: {len(FULL_LOOKUP)}")

# def _resolve_region(state_field, city, description, capability, name, address_line1, lookup_json):
#     p1 = _norm_region_field(state_field)
#     if p1:
#         return p1
#     lookup = json.loads(lookup_json)
#     city_l = str(city or "").strip().lower()
#     if city_l and city_l not in ("null","none",""):
#         if city_l in lookup:
#             return lookup[city_l]
#         for key, val in lookup.items():
#             if key and (key in city_l or city_l in key):
#                 return val
#     combined_text = " ".join(
#         str(x).lower()
#         for x in [description, capability, name, address_line1]
#         if x and str(x).strip().lower() not in ("null","none","")
#     )
#     for signal, region in REGION_TEXT_SIGNALS:
#         if signal in combined_text:
#             return region
#     return "Unknown"

# resolve_region_udf = F.udf(
#     lambda sf, cy, de, ca, nm, a1: _resolve_region(sf, cy, de, ca, nm, a1, _LOOKUP_JSON),
#     StringType()
# )

# # COMMAND ----------
# # DBTITLE 1,Step 12 — Fast Geocoding (SIL-1: replaces broken Nominatim)
# # All 202 silver rows had lat=7.9465/lon=-1.0232 because Nominatim always fell back.
# # Replaced with a curated city→coords dictionary (instant, zero rate limits).

# GHANA_CITY_COORDS = {
#     # Greater Accra
#     "accra":(5.6037,-0.1870),"tema":(5.6698,-0.0166),"adenta":(5.6888,-0.1695),
#     "ashaiman":(5.6877,-0.0291),"madina":(5.6832,-0.1637),"east legon":(5.6317,-0.1614),
#     "north legon":(5.6478,-0.1756),"teshie":(5.5837,-0.1078),"adabraka":(5.5634,-0.2108),
#     "dansoman":(5.5500,-0.2500),"kaneshie":(5.5567,-0.2231),"achimota":(5.6333,-0.2167),
#     "lapaz":(5.6057,-0.2507),"spintex":(5.6540,-0.1010),"osu":(5.5563,-0.1711),
#     "jamestown":(5.5330,-0.2040),"labadi":(5.5528,-0.1388),"legon":(5.6509,-0.1872),
#     "ridge":(5.5668,-0.1884),"kwabenya":(5.6744,-0.2165),"dome":(5.6501,-0.2354),
#     "awoshie":(5.6043,-0.2726),"haatso":(5.6584,-0.2048),"weija":(5.5700,-0.3222),
#     # Ashanti
#     "kumasi":(6.6885,-1.6244),"obuasi":(6.2034,-1.6718),"ejisu":(6.6667,-1.4833),
#     "bekwai":(6.4500,-1.5667),"mampong":(7.0667,-1.4000),"tafo":(6.7167,-1.6167),
#     "abuakwa":(6.7167,-1.5500),"drobonso":(6.6833,-1.4000),"buokrom":(6.7167,-1.5167),
#     "atonsu":(6.6667,-1.5500),"jacobu":(6.3500,-1.3667),"santasi":(6.6167,-1.6500),
#     "donyina":(6.6500,-1.4333),"offinso":(7.2667,-1.6667),"konongo":(6.6167,-1.2167),
#     # Western
#     "takoradi":(4.8845,-1.7554),"sekondi":(4.9392,-1.7050),"tarkwa":(5.3000,-1.9833),
#     "axim":(4.8703,-2.2403),"prestea":(5.4333,-2.1500),"bogoso":(5.5333,-1.9833),
#     "aboadze":(4.8667,-1.8333),"apremdo":(4.9500,-1.8167),"kwesimintsim":(4.9167,-1.8000),
#     # Western North
#     "bibiani":(6.4667,-2.3167),"sefwi wiawso":(6.2000,-2.4833),"juaboso":(6.1167,-2.8000),
#     # Eastern
#     "koforidua":(6.0940,-0.2596),"nkawkaw":(6.5500,-0.7833),"suhum":(6.0500,-0.4500),
#     "akim oda":(5.9333,-0.9833),"asamankese":(5.8667,-0.6667),"mpraeso":(6.5833,-0.7500),
#     # Central
#     "cape coast":(5.1053,-1.2466),"winneba":(5.3500,-0.6333),"saltpond":(5.2000,-1.0500),
#     "swedru":(5.5333,-0.7000),"mankessim":(5.2667,-1.0167),"kasoa":(5.5333,-0.4167),
#     "assin fosu":(5.5667,-1.2000),"agona nkwanta":(5.1333,-1.7167),
#     # Volta
#     "ho":(6.6008,0.4713),"hohoe":(7.1500,0.4667),"keta":(5.9167,0.9833),
#     "aflao":(6.1167,1.1833),"kpandu":(7.0000,0.3000),"sogakope":(5.8833,0.5833),
#     # Oti
#     "dambai":(8.0667,0.1833),"nkwanta":(8.1667,0.5167),
#     # Northern
#     "tamale":(9.4075,-0.8533),"yendi":(9.4500,-0.0167),"savelugu":(9.6333,-0.8167),
#     "walewale":(10.3500,-0.7833),
#     # Savannah
#     "salaga":(8.5500,-0.5167),"damongo":(9.0833,-1.8167),"bole":(9.0333,-2.4833),
#     # North East
#     "nalerigu":(10.6167,-0.3667),"bimbila":(9.8833,0.0500),
#     # Bono East
#     "techiman":(7.5841,-1.9364),"kintampo":(8.0500,-1.7000),"atebubu":(7.7500,-0.9833),
#     "nkoranza":(7.5500,-1.7000),
#     # Brong-Ahafo
#     "sunyani":(7.3349,-2.3266),"berekum":(7.4574,-2.5879),"wenchi":(7.7500,-2.1000),
#     "dormaa ahenkro":(7.3000,-2.8000),"banda":(7.8500,-2.4000),
#     # Upper East
#     "bolgatanga":(10.7835,-0.8514),"navrongo":(10.9000,-1.0833),
#     "bawku":(11.0500,-0.2333),"zebilla":(10.8667,-0.5667),
#     # Upper West
#     "wa":(10.0607,-2.5099),"lawra":(10.6333,-2.9000),
#     "jirapa":(10.5833,-2.7833),"nandom":(10.8667,-2.8167),
#     # Ahafo
#     "goaso":(6.7833,-2.5167),"bechem":(7.1667,-2.3333),"kukuom":(7.0167,-2.4000),
#     "acherensua":(7.3500,-2.4000),
# }


# REGION_CENTROIDS = {
#     "Greater Accra":(5.6037,-0.1870),"Ashanti":(6.6885,-1.6244),
#     "Western":(4.9500,-2.0000),"Central":(5.5000,-1.1000),
#     "Eastern":(6.5000,-0.3000),"Northern":(9.5000,-1.0000),
#     "Upper East":(10.7500,-0.7500),"Upper West":(10.2500,-2.5000),
#     "Volta":(6.8000,0.3000),"Brong-Ahafo":(7.5000,-2.0000),
#     "Bono East":(7.8000,-1.5000),"Ahafo":(7.3000,-2.7000),
#     "Western North":(6.5000,-2.7000),"Oti":(8.0000,0.0000),
#     "Savannah":(9.0000,-1.5000),"North East":(10.5000,-0.5000),
#     "Unknown":(7.9465,-1.0232),
# }



# # def _fast_geocode(city: Optional[str], region: Optional[str]) -> tuple:
# #     """
# #     Fast geocoding: city lookup → region centroid → Ghana centre.
# #     No external API calls, no rate limits. Replaces broken Nominatim approach.
# #     Returns (lat, lon, source).
# #     """
# #     city_k = str(city or "").strip().lower()
# #     if city_k and city_k not in ("null","none",""):
# #         if city_k in GHANA_CITY_COORDS:
# #             v = GHANA_CITY_COORDS[city_k]
# #             return (v[0], v[1], "city_lookup")
# #         # Partial match
# #         for k, v in GHANA_CITY_COORDS.items():
# #             if k and (k in city_k or (len(city_k) > 4 and city_k in k)):
# #                 return (v[0], v[1], "city_partial")
# #     region_k = str(region or "").strip()
# #     if region_k in REGION_CENTROIDS:
# #         v = REGION_CENTROIDS[region_k]
# #         return (v[0], v[1], "region_centroid")
# #     return (7.9465, -1.0232, "country_fallback")

# def _geopy_geocode(city: Optional[str], region: Optional[str]) -> tuple:
#     """
#     Geocode using geopy (Nominatim)
#     Priority:
#       1. City + Region + Ghana
#       2. Region + Ghana
#       3. Fallback (Ghana centroid)
#     """

#     key = f"{city}|{region}".lower()

#     # ---- CACHE HIT ----
#     if key in _geo_cache:
#         return _geo_cache[key]

#     try:
#         # ---- Query 1: City + Region ----
#         if city and str(city).strip().lower() not in ("null", "none", ""):
#             query = f"{city}, {region}, Ghana"
#             loc = geocode(query)
#             if loc:
#                 result = (loc.latitude, loc.longitude, "geopy_city")
#                 _geo_cache[key] = result
#                 return result

#         # ---- Query 2: Region ----
#         if region and str(region).strip().lower() not in ("null", "none", ""):
#             query = f"{region}, Ghana"
#             loc = geocode(query)
#             if loc:
#                 result = (loc.latitude, loc.longitude, "geopy_region")
#                 _geo_cache[key] = result
#                 return result

#     except Exception as e:
#         pass  # silent fail → fallback

#     # ---- Fallback ----
#     result = (7.9465, -1.0232, "fallback")
#     _geo_cache[key] = result
#     return result

# geocode_udf_lat = F.udf(lambda c, r: _geopy_geocode(c, r)[0], DoubleType())
# geocode_udf_lon = F.udf(lambda c, r: _geopy_geocode(c, r)[1], DoubleType())
# geocode_udf_src = F.udf(lambda c, r: _geopy_geocode(c, r)[2], StringType())

# # COMMAND ----------
# # DBTITLE 1,Step 13 — Apply All Transformations

# # 13a. Numerics
# def safe_int_col(col_name, pattern=r"(\d+)"):
#     return F.when(
#         F.col(col_name).isNull() | (F.regexp_extract(F.col(col_name), pattern, 1) == ""),
#         F.lit(None).cast(IntegerType())
#     ).otherwise(F.regexp_extract(F.col(col_name), pattern, 1).cast(IntegerType()))

# silver = (
#     silver
#     .withColumn("number_doctors_int",  safe_int_col("numberdoctors"))
#     .withColumn("capacity_int",         safe_int_col("capacity"))
#     .withColumn("area_int",             safe_int_col("area"))
#     .withColumn("year_established_int", safe_int_col("yearestablished", r"(\d{4})"))
#     # SIL-8: acceptsVolunteers defaults to False (no nulls)
#     .withColumn("accepts_volunteers_bool",
#         F.when(F.lower(F.col("acceptsvolunteers")).isin("true","1","yes"), True)
#          .when(F.lower(F.col("acceptsvolunteers")).isin("false","0","no"), False)
#          .otherwise(F.lit(False)))
#     .withColumn("pk_unique_id_int",
#         F.when(F.col("pk_unique_id").isNull() | (F.trim(F.col("pk_unique_id"))==""),
#                F.lit(None).cast(IntegerType()))
#          .otherwise(F.regexp_extract(
#              F.regexp_replace(F.col("pk_unique_id"), r"\.0$",""), r"(\d+)",1
#          ).cast(IntegerType())))
# )

# # 13b. Enrich capacity/doctors from free-text (overcomes sparse structured fields)
# silver = (
#     silver
#     .withColumn("capacity_int",
#         capacity_udf(F.col("capacity").cast(StringType()),
#                      F.col("description"),
#                      F.col("capability").cast(StringType())))
#     .withColumn("number_doctors_int",
#         doctors_udf(F.col("numberdoctors").cast(StringType()),
#                     F.col("description"),
#                     F.col("capability").cast(StringType())))
# )

# # 13c. Facility + org types
# silver = (
#     silver
#     .withColumn("facility_type_clean",
#         infer_facility_type_udf(
#             F.col("facilitytypeid"), F.col("name"),
#             F.col("description"), F.col("capability"),
#             F.col("organization_type")))
#     .withColumn("organization_type_clean",
#         heuristic_org_type_udf(
#             F.col("organization_type"), F.col("name"), F.col("source_url")))
# )

# # 13d. City + Region
# silver = (
#     silver
#     .withColumn("city_clean",
#         extract_city_udf(F.col("address_city"), F.col("address_line1"), F.col("name")))
#     .withColumn("region_normalised",
#         resolve_region_udf(
#             F.col("address_stateorregion"), F.col("city_clean"),
#             F.col("description"), F.col("capability").cast(StringType()),
#             F.col("name"), F.col("address_line1")))
# )

# # 13e. Geocoding (SIL-1: fast city lookup, no Nominatim)
# silver = (
#     silver
#     .withColumn("latitude",    geocode_udf_lat(F.col("city_clean"), F.col("region_normalised")))
#     .withColumn("longitude",   geocode_udf_lon(F.col("city_clean"), F.col("region_normalised")))
#     .withColumn("geo_source",  geocode_udf_src(F.col("city_clean"), F.col("region_normalised")))
# )

# print("✅ All structural transformations applied")

# # COMMAND ----------
# # DBTITLE 1,Step 14 — Content Flags (SIL-5)

# silver = (
#     silver
#     .withColumn("has_procedures",   F.when(F.size(F.col("procedure_parsed"))  > 0, True).otherwise(False))
#     .withColumn("has_equipment",    F.when(F.size(F.col("equipment_parsed"))   > 0, True).otherwise(False))
#     .withColumn("has_capabilities", F.when(F.size(F.col("capability_parsed")) > 0, True).otherwise(False))
#     .withColumn("has_specialties",  F.when(F.size(F.col("specialties_parsed"))> 0, True).otherwise(False))
#     .withColumn("has_description",
#         F.when(F.col("description").isNotNull() &
#                (F.length(F.trim(F.col("description"))) > 30), True).otherwise(False))
#     .withColumn("has_contact",
#         F.when((F.size(F.col("phone_numbers_parsed")) > 0) |
#                (F.col("email").isNotNull() & (F.col("email") != "")), True).otherwise(False))
#     .withColumn("procedure_count",  F.size(F.col("procedure_parsed")))
#     .withColumn("equipment_count",  F.size(F.col("equipment_parsed")))
#     .withColumn("capability_count", F.size(F.col("capability_parsed")))
#     .withColumn("specialty_count",  F.size(F.col("specialties_parsed")))
# )

# # COMMAND ----------
# # DBTITLE 1,Step 15 — doc_text + is_rag_ready (SIL-4)
# # These columns are REQUIRED by 05_rag_build_index.py for FAISS embedding.

# silver = (
#     silver
#     .withColumn("doc_text",
#         F.trim(F.regexp_replace(
#             F.concat_ws(" | ",
#                 F.coalesce(F.col("description"),          F.lit("")),
#                 F.coalesce(F.col("organizationdescription"), F.lit("")),
#                 F.array_join(F.col("capability_parsed"),  " "),
#                 F.array_join(F.col("procedure_parsed"),   " "),
#                 F.array_join(F.col("equipment_parsed"),   " "),
#                 F.array_join(F.col("specialties_parsed"), " "),
#                 F.coalesce(F.col("missionstatement"),     F.lit("")),
#             ),
#             r"\s{2,}", " "
#         )))
#     .withColumn("doc_text_length", F.length(F.col("doc_text")))
#     .withColumn("is_rag_ready",
#         F.col("doc_text_length") >= 80)   # at least 80 chars of meaningful text
# )

# rag_ready_ct = silver.filter(F.col("is_rag_ready")).count()
# print(f"RAG-ready rows (doc_text ≥ 80 chars): {rag_ready_ct:,}")

# # COMMAND ----------
# # DBTITLE 1,Step 16 — Deduplication (unique_id + cross-source name+city)

# RICHNESS_SCALAR = [
#     "number_doctors_int","capacity_int","area_int","year_established_int",
#     "address_line1","city_clean","email","officialwebsite",
#     "description","facilitytypeid","operatortypeid",
# ]
# RICHNESS_ARRAYS = [f"{c}_parsed" for c in ["specialties","procedure","equipment","capability"]]

# richness_expr = (
#     sum(
#         F.when(F.col(c).isNotNull() & (F.col(c).cast(StringType()) != ""), 1).otherwise(0)
#         for c in RICHNESS_SCALAR if c in silver.columns
#     )
#     + sum(
#         F.when(F.size(F.col(c)) > 0, F.size(F.col(c))).otherwise(0)
#         for c in RICHNESS_ARRAYS if c in silver.columns
#     )
# )
# silver = silver.withColumn("_row_richness", richness_expr)

# # Pass 1: deduplicate on unique_id (keep richest row)
# w1 = Window.partitionBy("unique_id").orderBy(F.desc("_row_richness"), F.desc("ingested_at"))
# silver = silver.withColumn("_r1", F.row_number().over(w1)).filter(F.col("_r1")==1).drop("_r1")
# after_uid = silver.count()
# print(f"After unique_id dedup: {after_uid:,}")

# # Pass 2: cross-source dedup on normalised name + city
# silver = (
#     silver
#     .withColumn("_name_norm",
#         F.lower(F.trim(F.regexp_replace(F.col("name"), r"[^a-z0-9\s]",""))))
#     .withColumn("_city_norm",
#         F.lower(F.trim(F.coalesce(F.col("city_clean"), F.lit("")))))
# )
# w2 = Window.partitionBy("_name_norm","_city_norm").orderBy(F.desc("_row_richness"), F.desc("ingested_at"))
# silver = (
#     silver.withColumn("_r2", F.row_number().over(w2))
#           .filter(F.col("_r2")==1)
#           .drop("_r2","_name_norm","_city_norm","_row_richness")
# )
# after_cross = silver.count()
# print(f"After cross-source dedup: {after_cross:,}  (removed {after_uid - after_cross:,})")

# # COMMAND ----------
# # DBTITLE 1,Step 17 — Data Completeness Score (SIL-7)

# COMPLETENESS_WEIGHTS = {
#     "number_doctors_int"  : 1.0,
#     "capacity_int"        : 1.0,
#     "specialties_parsed"  : 1.0,
#     "procedure_parsed"    : 1.0,
#     "equipment_parsed"    : 1.0,
#     "capability_parsed"   : 1.0,
#     "city_clean"          : 0.5,
#     "region_normalised"   : 0.5,   # 0 if Unknown
#     "officialwebsite"     : 0.5,
#     "email"               : 0.5,
#     "phone_numbers_parsed": 0.5,
#     "address_line1"       : 0.5,
#     "description"         : 1.0,
#     "facility_type_clean" : 0.5,
#     "operatortypeid"      : 0.5,
#     "year_established_int": 0.3,
#     "is_rag_ready"        : 0.7,   # SIL-7: RAG-readiness is a strong quality signal
# }
# _max_score = sum(COMPLETENESS_WEIGHTS.values())

# _parts = []
# for f, w in COMPLETENESS_WEIGHTS.items():
#     if f not in silver.columns:
#         continue
#     if f == "region_normalised":
#         part = F.when(F.col(f).isNull() | F.col(f).isin("Unknown",""), F.lit(0.0)).otherwise(F.lit(w))
#     elif f == "is_rag_ready":
#         part = F.when(F.col(f) == True, F.lit(w)).otherwise(F.lit(0.0))
#     elif "parsed" in f:
#         part = F.when(F.size(F.col(f)) > 0, F.lit(w)).otherwise(F.lit(0.0))
#     else:
#         part = F.when(
#             F.col(f).isNotNull() & ~F.col(f).cast(StringType()).isin("","null","None","Unknown"),
#             F.lit(w)
#         ).otherwise(F.lit(0.0))
#     _parts.append(part)

# completeness_sum = reduce(op_module.add, _parts) if _parts else F.lit(0.0)
# silver = silver.withColumn(
#     "data_completeness_score",
#     (completeness_sum / F.lit(_max_score)).cast(FloatType())
# )

# avg_score = silver.agg(F.avg("data_completeness_score")).collect()[0][0]
# print(f"Average data completeness score: {avg_score:.3f} ({avg_score*100:.1f}%)")

# # COMMAND ----------
# # DBTITLE 1,Step 18 — Write Silver Delta Table

# (
#     silver.write
#     .format("delta")
#     .mode("overwrite")
#     .option("overwriteSchema", "true")
#     .option("delta.autoOptimize.optimizeWrite", "true")
#     .option("delta.autoOptimize.autoCompact", "true")
#     .saveAsTable(cfg.SILVER)
# )

# final = spark.table(cfg.SILVER)
# print(f"\n✅ Silver table written: {cfg.SILVER}")
# print(f"   Rows    : {final.count():,}")
# print(f"   Columns : {len(final.columns)}")

# # COMMAND ----------
# # DBTITLE 1,Step 19 — Quality Report

# s     = spark.table(cfg.SILVER)
# total = s.count()

# print("="*65)
# print(f"SILVER QUALITY REPORT  ({total:,} rows)")
# print("="*65)

# checks = [
#     ("unique_id",            "scalar", "Unique ID present"),
#     ("name",                 "scalar", "Name present"),
#     ("city_clean",           "scalar", "City present"),
#     ("region_normalised",    "region", "Region normalised (non-Unknown)"),
#     ("latitude",             "geo",    "Geocoded (non-fallback)"),
#     ("facility_type_clean",  "scalar", "Facility type set"),
#     ("organization_type_clean","scalar","Org type set"),
#     ("number_doctors_int",   "scalar", "Doctor count present"),
#     ("capacity_int",         "scalar", "Bed capacity present"),
#     ("description",          "scalar", "Description present"),
#     ("specialties_parsed",   "array",  "Specialties non-empty"),
#     ("capability_parsed",    "array",  "Capability non-empty"),
#     ("is_rag_ready",         "bool",   "RAG-ready (doc_text ≥ 80 chars)"),
#     ("data_completeness_score","thresh","Completeness score ≥ 0.3"),
# ]

# for col_name, check_type, label in checks:
#     if col_name not in s.columns:
#         print(f"  {'MISSING':8} {label}"); continue
#     if check_type == "region":
#         ct = s.filter(F.col(col_name).isNotNull() & ~F.col(col_name).isin("Unknown","")).count()
#     elif check_type == "array":
#         ct = s.filter(F.size(F.col(col_name)) > 0).count()
#     elif check_type == "thresh":
#         ct = s.filter(F.col(col_name) >= 0.3).count()
#     elif check_type == "bool":
#         ct = s.filter(F.col(col_name) == True).count()
#     elif check_type == "geo":
#         ct = s.filter(F.col("geo_source") != "country_fallback").count()
#     else:
#         ct = s.filter(F.col(col_name).isNotNull() & (F.col(col_name).cast(StringType())!="")).count()
#     pct = ct / total * 100
#     bar = "█" * int(pct/5)
#     print(f"  {label:<48} {pct:5.1f}%  {bar}")

# print("\nGeo source distribution:")
# s.groupBy("geo_source").count().orderBy(F.desc("count")).show()
# print("\nRegion distribution:")
# s.groupBy("region_normalised").count().orderBy(F.desc("count")).show(20)
# print("\nOrg type distribution:")
# s.groupBy("organization_type_clean").count().orderBy(F.desc("count")).show()
# print("\nFacility type distribution:")
# s.groupBy("facility_type_clean").count().orderBy(F.desc("count")).show()

In [0]:
%sql
DESCRIBE TABLE virtue_foundation.ghana.silver_facilities_cleaned 

In [0]:
%sql
SHOW CREATE TABLE virtue_foundation.ghana.silver_facilities_cleaned